In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import os
%matplotlib widget
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':8})
pd.options.display.max_columns = 200
pd.options.display.max_rows = 300
pd.options.display.max_seq_items = 2000
# from tqdm import tqdm
from datetime import timedelta
import re
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, get_metadata, read_in_routine
from tqdm import tqdm 

from medications import *

In [2]:
def opioid_benzo_antipsychotic_list():
    
    opioids_meds = ['fentanyl','hydromorphone','morphine','remifentanil','tramadol',
     'tapentadol','oxymorphone','oxycodone','hydrocodone','methadone',
     'propoxyphene','meperidine','codeine','alfentanil','sufentanil']

    benzos_meds = ['diazepam','lorazepam','midazolam','clonazepam','diazepam','estazolam',
     'flunitrazepam','lorazepam','midazolam','nitrazepam','oxazepam',
     'triazolam','temazepam','chlordiazepoxide','alprazolam','clobazam',
     'clorazepate','etizolam', 'propofol', 'dexmedetomidine']

    antipsychotics_meds = ['olanzapine','clozapine','thiothixene','haloperidol','fluphenazine',
     'prochlorperazine','trifluoperazine','loxapine','quetiapine','asenapine']
    
    return opioids_meds, benzos_meds, antipsychotics_meds


In [3]:
# SET PATHS FOR THE FILES NEEDED:

# project = 'sample'
project = 'icu-sleep'
project = 'covid-risk-pred-ncc'
project = 'covid_sedation'
# project = 'covid_delirium'

if project == 'icu-sleep':
    cohort_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/LIST.csv'
    save_path_meds_filtered_for_icu_sleep_encounter = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_meds_encounter.csv'
    medication_categories_path = 'medication_categories.xlsx'
    edw_data_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_meds.csv'
    adt_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_adt.csv'
    flowsheets_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_flowsheets.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_all_output_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/medications_all_timeseries'
    
elif project == 'covid-risk-pred-ncc':
    cohort_path = None
    save_path_meds_filtered_for_icu_sleep_encounter = None    # not necessary for this project
    medication_categories_path = 'Partners_COVID_clinical_trials.csv'
    edw_data_path = '/media/ssd2/Covid19_Risk_Prediction/Medication_all_covidflagged_mar10tojun30.csv'
    adt_path = None # '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_adt.csv'
    flowsheets_path = None # '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_flowsheets.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_all_output_dir = f'/media/ssd2/Covid19_Risk_Prediction/{project}/medications_all_timeseries'

elif project == 'covid_sedation':
    edw_data_path = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/edw_covid_sedation_meds.csv'
    cohort_path = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/cohort_01012020.csv'
    medication_categories_path = None
    adt_path = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/edw_covid_sedation_adt.csv'
    flowsheets_path = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/edw_covid_sedation_flowsheets.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_all_output_dir = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries'

elif project == 'covid_delirium':
    edw_data_path = '/media/ssd2/Covid19_Delirium/edw_covid_delirium_meds.csv'
    cohort_path = '/media/ssd2/Covid19_Delirium/covid_delirium_mrns.xlsx'
    medication_categories_path = None
    adt_path = '/media/ssd2/Covid19_Delirium/edw_covid_delirium_adt.csv'
    flowsheets_path = '/media/ssd2/Covid19_Delirium/edw_covid_delirium_flowsheets.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_all_output_dir = '/media/ssd2/Covid19_Delirium/medications_all_timeseries'

elif project == 'sample':
    cohort_path = None
    save_path_meds_filtered_for_icu_sleep_encounter = None
    medication_categories_path = 'med_dd.xlsx'
    edw_data_path = '/media/mad3/Projects/covid_data/EDW/edw_covid_sample_meds.csv'
    adt_path = '/media/mad3/Projects/covid_data/EDW/edw_covid_sample_adt.csv'
    flowsheets_path = '/media/mad3/Projects/covid_data/EDW/edw_covid_sample_flowsheets.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_all_output_dir = '/media/mad3/Projects/covid_data/EDW/medications_all_timeseries'

    
if not os.path.exists(meds_all_output_dir):
    os.makedirs(meds_all_output_dir)
    


In [4]:
# get Med Categories and a list of all medications that are to be studied:

if project in ['icu-sleep', 'sample']:
    med_categories = pd.read_excel(medication_categories_path)
    med_categories.columns = [x.replace('-','_').replace('\n','').replace(': ','_') for x in med_categories.columns]
    for col in med_categories.columns:
        med_categories[col] = med_categories[col].str.lower()
    med_categories.head(3)

    if project == 'icu-sleep':
        meds_to_study = [x.lower().strip(' ') for x in np.array(med_categories).flatten() if not pd.isna(x)]

    elif project == 'sample':
        meds_to_study = list(med_categories.generic_name.str.replace('\xa0', '').values)
        
elif project == 'covid-risk-pred-ncc':
    
    med_categories = pd.read_csv(medication_categories_path, sep='\t')
    med_categories['medname'] = med_categories['generic_name'] + med_categories['COVID treatment type'].apply(lambda x: ' ; Placebo' if 'placebo' in x.lower() else '')
    meds_to_study = list(med_categories['medname'].str.lower())
    
elif project == 'covid_sedation':
    med_list_type = 'name_erx'
    meds_to_study = {
    'Quetiapine': [2183, 21824],
    'Haloperidol': [3585, 3584, 10162, 3578, 3581, 3583, 4004082788],
    'Olanzapine': [28159, 38263, 21057, 17936],
    'Risperidone': [35686, 35687, 17377],
    'Ziprasidone': [33175],
    'Aripiprazole': [76823, 70306, 36438, 34370],
    'Chlorpromazine': [40042091, 4004081579, 4004081580, 4004081581, 4004082787, 4004081582, 1649, 1653, 1654, 1656],
    'Valproic acid': [40042145, 4004081267, 4004081266, 20887, 127503, 8429, 27631, 2551, 2553, 34418, 81426],
    'Phenobarbital': [6215, 6212, 6219, 6220, 6004200028, 40042138, 4004082809, 4004082808, 4004082017, 6221, 6224],
    'Gabapentin': [18309, 29169, 18308, 160018, 18307, 40042051, 5004201003],
    'Ketamine': [4236, 4237, 4238, 701576, 40042242, 701114, 702142, 40042224, 700038, 40042151, 1004200015],
    'Dexmedetomidine': [27103, 181527, 7006048, 6004080205, 4004081395, 1004200072, 40042097],
    'Clonidine': [27505, 27506, 27507, 1755, 1756],
    'Isoflurane': [127796],
    'Propofol': [11150, 701735, 40042190, 40042189, 60042003, 701734],
    'Midazolam': [172732, 172731, 10608, 700622, 709150, 6004200010, 40042228, 7005111, 700041, 1004080126, 1004080124, 1004080127, 1004080125, 1004200076, 40042042, 701800, 7000097, 40042195],
    'Diazepam': [2404, 2405, 2402, 114047, 2401, 106278],
    'Lorazepam': [4572, 4573, 10467, 4571, 4004229, 40042153],
    'Clonazepam': [35626, 9637, 9638, 700122, 700994],
    'Alprazolam': [324, 325],
    'Chlordiazepoxide': [1623, 1624],
    'Methadone': [4953, 10546, 15996, 4954, 4952],
    'Fentanyl': [709111, 3037,27908, 41382, 27905, 27906, 27907, 700629, 701610, 700646, 40042232, 700001, 701220, 40042108, 702138, 5004200025],
    'Hydromorphone': [192972, 3757, 10224, 700102, 701604, 709542, 702131, 702146, 162068, 127812, 126597, 119834, 10225, 3760, 3758, 118595, 10226, 701612, 700660, 702147, 701609, 40042244, 40042056, 40042117],
    'Morphine': [29464, 15852, 770099, 27392, 30851, 5176, 5168, 27390, 5178, 5169, 116036, 5170, 160280, 116697, 5172, 160281, 185637, 5173, 116966, 116040, 5175, 700618, 702151, 40042230, 159236, 701606, 701804, 40042126, 709538, 702141, 40042125, 40042295],
    'Oxycodone': [87822, 10812, 10814, 10813],
    'Hydrocodone': [34505, 70832],
    'Tramadol': [14632],
    'Meperidine': [123225, 4903, 4904, 119043, 40042120],
    'Rocuronium': [700794, 707434, 95811, 125950, 701594, 40042074, 701929, 40042199],
    'Vecuronium': [700762, 701741, 6004200012, 11634, 1004200084, 40042066, 700485, 701739, 701740, 4004082804, 500420052, 40042279]
    }
    

elif project == 'covid_delirium':
    med_list_type = 'name'
    meds_to_study = ['Quetiapine',
    'Haloperidol',
    'Olanzapine',
    'Risperidone',
    'Ziprasidone',
    'Aripiprazole',
    'Chlorpromazine',
    'Valproic acid',
    'Phenobarbital',
    'Gabapentin',
    'Ketamine',
    'Dexmedetomidine',
    'Clonidine',
    'Isoflurane',
    'Propofol',
    'Midazolam',
    'Diazepam',
    'Lorazepam',
    'Clonazepam',
    'Alprazolam',
    'Chlordiazepoxide',
    'Methadone',
    'Fentanyl',
    'Hydromorphone',
    'Morphine',
    'Oxycodone',
    'Hydrocodone',
    'Tramadol',
    'Meperidine',
    'Cisatracurium',
    'Rocuronium',
    'Vecuronium',
    'Modafinil',
    'Methylphenidate',
    'Amantadine',
    'Caffeine',
    'Melatonin']
    meds_to_study = [x.lower() for x in meds_to_study]

In [5]:
if med_list_type == 'name_erx':
    meds_to_study_df = pd.DataFrame(columns = ['med_name', 'erx'])
    i = 0
    for key in meds_to_study.keys():
        values = meds_to_study[key]
        for value in values:
            i += 1
            meds_to_study_df.loc[i, 'med_name'] = key
            meds_to_study_df.loc[i, 'erx'] = value
    meds_to_study_df

In [6]:
if project in ['icu-sleep', 'sample']:
    opioids_meds, benzos_meds, antipsychotics_meds = opioid_benzo_antipsychotic_list()
    for x in opioids_meds + benzos_meds + antipsychotics_meds:
        if x not in meds_to_study:
            meds_to_study.append(x)
    meds_to_study = pd.unique(meds_to_study)

In [7]:
if cohort_path is not None:
    if project == 'icu-sleep':
        cohort = pd.read_csv(cohort_path)
        cohort.dropna(subset=['MRN:'], inplace=True)
        cohort.loc[cohort['Study ID:']=='31', 'MRN:'] = 1876849
        cohort.loc[cohort['Study ID:']=='15', 'MRN:'] = 1479357
        cohort = cohort[cohort['Study ID:']!='98a']
        cohort['study_id'] = cohort['Study ID:'].apply(lambda x: str(x).zfill(3))
        cohort['mrn'] = cohort['MRN:'].apply(lambda x: int(x))
        cohort['datetime'] = pd.to_datetime(cohort['First Day Enrolled in Study:'], infer_datetime_format=1)
        cohort.head(2)
    elif project == 'covid_sedation':
        cohort = pd.read_csv(cohort_path)
        
    elif project == 'covid_delirium':
        cohort = pd.read_excel(cohort_path)
        cohort.rename({'admisison_date': 'Admission', 'discharge_date': 'Discharge'}, axis=1, inplace=True)
        
elif adt_path is not None:
    adt_data = pd.read_csv(adt_path)
    cohort = pd.DataFrame([])
    cohort['mrn'] = adt_data.MRN.unique()
#     cohort['datetime'] = ['2020-04-08', '2020-04-04']
#     cohort['datetime'] = pd.to_datetime(cohort['datetime'], infer_datetime_format=1)
    
else:
    cohort = pd.DataFrame([])

In [8]:
flowsheet_data = pd.read_csv(flowsheets_path)

# GCS:
gcs_data = flowsheet_data.loc[['glasgow' in x for x in flowsheet_data.FlowsheetMeasureNM.str.lower()]].copy()

In [9]:
if project == 'covid_sedation':
    
    flowsheet_data = pd.read_csv(flowsheets_path)
    flowsheet_data.drop_duplicates(subset=['PatientID', 'RecordedDTS', 'MeasureTXT', 'FlowsheetDataID'], inplace=True)
    
    # GCS:
    gcs_data = flowsheet_data.loc[['glasgow' in x for x in flowsheet_data.FlowsheetMeasureNM.str.lower()]].copy()
    gcs_data = gcs_data[gcs_data.FlowsheetMeasureNM == 'R CPN GLASGOW COMA SCALE SCORE']
    gcs_data['RecordedDTS'] = pd.to_datetime(gcs_data['RecordedDTS'], infer_datetime_format=1)
    gcs_data.sort_values(by = ['RecordedDTS'], inplace=True)
    gcs_data.set_index('RecordedDTS', inplace=True)    
    
    
    gcs_data_motor = flowsheet_data[flowsheet_data.FlowsheetMeasureNM == 'R CPN GLASGOW COMA SCALE BEST MOTOR RESPONSE']
    gcs_data_motor['RecordedDTS'] = pd.to_datetime(gcs_data_motor['RecordedDTS'], infer_datetime_format=1)
    gcs_data_motor.sort_values(by = ['RecordedDTS'], inplace=True)
    gcs_data_motor.set_index('RecordedDTS', inplace=True)        
    
    gcs_data_eye = flowsheet_data[flowsheet_data.FlowsheetMeasureNM == 'R CPN GLASGOW COMA SCALE EYE OPENING']
    gcs_data_eye['RecordedDTS'] = pd.to_datetime(gcs_data_eye['RecordedDTS'], infer_datetime_format=1)
    gcs_data_eye.sort_values(by = ['RecordedDTS'], inplace=True)
    gcs_data_eye.set_index('RecordedDTS', inplace=True)        
    
    gcs_data_verbal = flowsheet_data[flowsheet_data.FlowsheetMeasureNM == 'R CPN GLASGOW COMA SCALE BEST VERBAL RESPONSE']
    gcs_data_verbal['RecordedDTS'] = pd.to_datetime(gcs_data_verbal['RecordedDTS'], infer_datetime_format=1)
    gcs_data_verbal.sort_values(by = ['RecordedDTS'], inplace=True)
    gcs_data_verbal.set_index('RecordedDTS', inplace=True)        
    
    gcs_data_motor['MeasureTXT'] = gcs_data_motor.MeasureTXT.apply(lambda x: str(x).replace('P', '')).astype(float)
    gcs_data_eye['MeasureTXT'] = gcs_data_eye.MeasureTXT.apply(lambda x: str(x).replace('C', '')).astype(float)
    gcs_data_verbal['MeasureTXT'] = gcs_data_verbal.MeasureTXT.apply(lambda x: str(x).replace('T', '')).astype(float)
    
    gcs_data.rename({'MeasureTXT': 'GCS'}, axis=1, inplace=True)
    gcs_data_motor.rename({'MeasureTXT': 'GCS_Motor'}, axis=1, inplace=True)
    gcs_data_eye.rename({'MeasureTXT': 'GCS_Eye'}, axis=1, inplace=True)
    gcs_data_verbal.rename({'MeasureTXT': 'GCS_Verbal'}, axis=1, inplace=True)
    

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  from ipykernel import kernelapp as app
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  app.launch_new_instance()
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [10]:
# add weight to cohort

if project == 'covid_sedation':
    flowsheet_patient_id = 'PatientID'
    cohort_patient_id = 'PatientID'
elif project == 'covid_delirium':
    flowsheet_patient_id = 'MRN'
    cohort_patient_id = 'mrn'

weight_data = flowsheet_data[(flowsheet_data.FlowsheetMeasureNM=='WEIGHT/SCALE')].copy()
weight_data['weight_kg'] = (weight_data['MeasureTXT'].astype('float')/35.274).round(1)
weight_data['RecordedDTS'] = pd.to_datetime(weight_data['RecordedDTS'], infer_datetime_format=1)

cohort['Weight'] = np.nan

for jloc, row in cohort.iterrows():

    zno_tmp = row[cohort_patient_id]
    
    weight_data_mrn = weight_data[weight_data[flowsheet_patient_id] == zno_tmp]
    weight_data_mrn = weight_data_mrn[weight_data_mrn.RecordedDTS > '2020-02-01']
    weight = np.nanmedian(weight_data_mrn.weight_kg.astype(float))
    cohort.loc[jloc, 'Weight'] = weight


In [11]:
if project == 'covid_delirium': # for this project, we want to have summary dose for whole stay. i.e. we can add the dosages to the cohort table
    for med_tmp in meds_to_study:
        cohort[med_tmp.lower()] = 0
        
    cohort['opioids_sum'] = 0
    cohort['benzos_sum'] = 0
    cohort['antipsychotics_sum'] = 0

In [12]:
# add MRNs to cohort:
    
if project == 'covid_sedation':
    cohort['mrn'] = np.nan
    for jloc, row in cohort.iterrows():
        if sum(weight_data.PatientID == row.PatientID) > 0:
            cohort.loc[jloc, 'mrn'] = int(weight_data.loc[weight_data.PatientID == row.PatientID, 'MRN'].values[0])

In [13]:
if project in ['icu-sleep']:
    
    if adt_path is not None:

        adt_data = pd.read_csv(adt_path)
        adt_data['EffectiveDTS'] = pd.to_datetime(adt_data['EffectiveDTS'], infer_datetime_format=1)
        adt_data = adt_data.sort_values(by=['EffectiveDTS'])

    if flowsheets_path is not None:
        weight_data = pd.read_csv(flowsheets_path)
        weight_data = weight_data[(weight_data.FlowsheetMeasureNM=='WEIGHT/SCALE')]
        weight_data['weight_kg'] = (weight_data['MeasureTXT'].astype('float')/35.274).round(1)
        weight_data['RecordedDTS'] = pd.to_datetime(weight_data['RecordedDTS'], infer_datetime_format=1)
        # weight_data['EffectiveDTS'] = pd.to_datetime(adt_data['EffectiveDTS'], infer_datetime_format=1)
        # weight_data = adt_data.sort_values(by=['EffectiveDTS'])

        # add weight to cohort
        cohort['Weight'] = np.nan

        for jloc, row in cohort.iterrows():

            mrn_tmp = row.mrn

            enrollment_date =  pd.Timestamp(row['datetime'])
            weight_data_mrn = weight_data[weight_data.MRN == mrn_tmp]
            weight_data_mrn_post = weight_data_mrn[weight_data_mrn.RecordedDTS > enrollment_date]
            weight_data_mrn_pre = weight_data_mrn[weight_data_mrn.RecordedDTS < enrollment_date]

            if len(weight_data_mrn_post) > 0:
                cohort.loc[jloc, 'Weight'] = weight_data_mrn_post['weight_kg'].dropna().iloc[0]
            elif len(weight_data_mrn_pre) > 0:
                cohort.loc[jloc, 'Weight'] = weight_data_mrn_pre['weight_kg'].dropna().iloc[-1]
            else:
                print(f'NO WEIGHT FOUND FOR MRN {mrn_tmp}')

        # GCS:

        gcs_data = pd.read_csv(flowsheets_path)
        print(pd.unique([x for x in gcs_data.MeasureTXT.str.lower() if np.any(['rass' in str(x), 'sedation' in str(x), 'richmond' in str(x)])]))
        gcs_data = gcs_data.loc[['glasgow' in x for x in gcs_data.FlowsheetMeasureNM.str.lower()]].copy()
        gcs_data = gcs_data[gcs_data.FlowsheetMeasureNM == 'R CPN GLASGOW COMA SCALE SCORE']
        gcs_data['RecordedDTS'] = pd.to_datetime(gcs_data['RecordedDTS'], infer_datetime_format=1)
        gcs_data.sort_values(by = ['RecordedDTS'], inplace=True)
        gcs_data.set_index('RecordedDTS', inplace=True)
        gcs_data.head(2)

In [14]:
cohort

,index,PatientID,AdmissionDTS,DischargeDTS,First_ICU_Admit,First_ICU_Discharge,Second_ICU_Admit,Second_ICU_Discharge,Third_ICU_Admit,Third_ICU_Discharge,Fourth_ICU_Admit,Fourth_ICU_Discharge,Number_ICU_Admissions,Age; years,Hospital LOS; days,ICU LOS; days,OnsetDTS,Hospital,PatientEncounterID,MRN,BirthDTS,SexDSC,EthnicGroupDSC,PatientRaceDSC,Extubation,Intubation,CMO,DNR,DNI,Age,Weight,Height,PMH Neurological Disorder,If PMH neuro disorder; describe,Stroke,Coma,Clinical or Electrographc Seizure/Status Epilepticus,Neuroimaging,ECMO,Dialysis/CRRT,CMO.1,In Hospital Death (yes/no),Date of death,1stETTplacementDTS,1stETTremovalDTS,2ndETTplacementDTS,2ndETTremovalDTS,3rdETTplacementDTS,3rdETTremovalDTS,4thETTplacementDTS,4thETTremovalDTS,Duration of ETT Placement,InHospitalMortality,CVVH_HD,Baseline Opioid/BZD (no=0; prn=1; scheduled=2),Baseline Opioid/BZD Comment,Intubated > 24 h at OSH,Discharge Disposition,Discharge Opioids (no=0; prn =1; scheduled =2),Discharge BZDs,Discharge Rx,Unnamed: 61,Ventilation_Off_DTS,PaO2_Arterial_PreIntubation,PaO2_Arterial_PreIntubation_DTS,PaO2_Arterial_PostIntubation,PaO2_Arterial_PostIntubation_DTS,PaO2_Venous_PreIntubation,PaO2_Venous_PreIntubation_DTS,PaO2_Venous_PostIntubation,PaO2_Venous_PostIntubation_DTS,Intubation_DTS,Intubation_Duration,ROC_DTS,ROC_Binary,ROC_LOC,ROC_Day7,gcs_num,gcs_6_days,gcs_8_days,rass_4_5_days,Time_To_ROC,PaO2_minimum,pao2_num,PaO2_55,Cat_Days_PaO2_55,55_First_DTS,55_t_Admission,55_t_Intubation,55_t_ROC,PaO2_70,Cat_Days_PaO2_70,70_First_DTS,70_t_Admission,70_t_Intubation,70_t_ROC,PaO2_50,Cat_Days_PaO2_50,PaO2_51,Cat_Days_PaO2_51,PaO2_52,Cat_Days_PaO2_52,PaO2_53,Cat_Days_PaO2_53,PaO2_54,Cat_Days_PaO2_54,PaO2_56,Cat_Days_PaO2_56,PaO2_57,Cat_Days_PaO2_57,PaO2_58,Cat_Days_PaO2_58,PaO2_59,Cat_Days_PaO2_59,PaO2_60,Cat_Days_PaO2_60,PaO2_61,Cat_Days_PaO2_61,PaO2_62,Cat_Days_PaO2_62,PaO2_63,Cat_Days_PaO2_63,PaO2_64,Cat_Days_PaO2_64,PaO2_65,Cat_Days_PaO2_65,PaO2_66,Cat_Days_PaO2_66,PaO2_67,Cat_Days_PaO2_67,PaO2_68,Cat_Days_PaO2_68,PaO2_69,Cat_Days_PaO2_69,spo2_intubation,spo2_course,spo2_84,berlin_intubation_value,berlin_course_value,berlin_intubation,berlin_course,MAP_min_intubation,MAP_65_intubation,MAP_min_course,MAP_65_course,highest_peep,highest_plat,hgb_min,hgb_7,mrn
0,0,Z10002895,2020-05-04 15:09:00,2020-05-19 11:38:00,2020-05-06 17:23:00,2020-05-15 13:23:00,NaN,NaN,NaN,NaN,NaN,NaN,1,56.431866,14.853472,8.833333,2020-04-28 00:00:00,MGH,3305390010,10087535802,1963-12-13 00:00:00,Male,Not Hispanic or Latino,Other,Yes,Yes,No,No,No,56,95.30,172.72000,No,NaN,No,Yes,No,No,No,No,No,No,No,2020-05-06 18:05:00,2020-05-15 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,8.371528,0.0,0.0,0.0,NaN,0.0,Long Term Care,0.0,0.0,NaN,NaN,2020-05-14 15:00:00,NaN,NaN,223.0,2020-05-06 22:30:00,NaN,NaN,NaN,NaN,2020-05-06 18:05:00,7.871528,2020-05-14 04:00:00,1.0,1.0,0.0,184.0,8.0,13.0,7.0,7.413194,46.0,13.0,1.0,1.0,2020-05-10 01:50:00,5.445139,3.322917,4.090278,1.0,2.0,2020-05-08 01:27:00,3.429167,1.306944,6.106250,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,85.0,83.0,1.0,322.5,153.3,0.0,2.0,68.0,0.0,61.0,1.0,8.0,22.0,11.8,0.0,2848304.0
1,2,Z10082075,2020-06-12 21:22:00,2020-07-01 13:20:00,2020-06-12 21:22:00,2020-06-17 13:37:00,NaN,NaN,NaN,NaN,NaN,NaN,1,73.695590,18.665278,4.677083,2020-04-20 00:00:00,MGH,3303872830,10069650728,1946-10-21 00:00:00,Male,Not Hispanic or Latino,Black or African American,Yes,Yes,No,No,No,73,64.90,175.26000,Yes,Parkinsons.hx,No,Yes,No,Yes,No,No,No,No,No,2020-06-12 16:47:00,2020-06-14 10:26:00,NaN,NaN,NaN,NaN,NaN,NaN,1.735417,0.0,0.0,0.0,NaN,0.0,Skilled Nursing Facility,0.0,0.0,NaN,NaN,2020-06-14 10:00:00,NaN,NaN,131.0,2020-06-13 03:12:00,NaN,NaN,NaN,NaN,2020-06-12 16:47:00,1.717361,2020-06-16 18:00:00,1.0,1.0,0.0,162.0,19.0,19.0,1.0,4.050694,110.0,2.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [15]:
if project == 'covid_sedation':
  
    cohort['Admission'] = cohort['AdmissionDTS']
    cohort['Discharge'] = cohort['DischargeDTS']
    
    if 0:     # this is not needed any more, we have admission and discharge data.
        cohort['Admission'] = np.nan
        cohort['Discharge'] = np.nan

        for jloc, row in cohort.iterrows():

            mrn_tmp = row.mrn
            enrollment_date = pd.Timestamp(row['datetime'])
            enrollment_date = enrollment_date.replace(hour=23)
            adt_mrn = adt_data[adt_data.MRN == mrn_tmp]
            adt_mrn = adt_mrn.sort_values(by='EffectiveDTS')

            admissions = adt_mrn.loc[(adt_mrn.ADTEventTypeDSC=='Admission') & (adt_mrn.EffectiveDTS<enrollment_date), 'EffectiveDTS'].drop_duplicates().values
            admissions_post = adt_mrn.loc[(adt_mrn.ADTEventTypeDSC=='Admission') & (adt_mrn.EffectiveDTS>enrollment_date), 'EffectiveDTS'].drop_duplicates().values
            discharges = adt_mrn.loc[(adt_mrn.ADTEventTypeDSC=='Discharge') & (adt_mrn.EffectiveDTS>enrollment_date), 'EffectiveDTS'].drop_duplicates().values
            admission = admissions[-1]
            discharge = pd.Timestamp(admission) + timedelta(days=60)

            cohort['Admission'].loc[jloc] = admission
            cohort['Discharge'].loc[jloc] = discharge

else:
    
    if ('Admission' in cohort.columns) | ('Discharge' in cohort.columns):
        print('Admission or Discharge already in cohort. Nothing computed here.')
    else:
        cohort['Admission'] = np.nan
        cohort['Discharge'] = np.nan

        min_admission = '2020-02-01'

        for jloc, row in cohort.iterrows():

            mrn_tmp = row.mrn
            adt_mrn = adt_data[adt_data.MRN == mrn_tmp]
            adt_mrn = adt_mrn.sort_values(by='EffectiveDTS')
            admissions = adt_mrn.loc[(adt_mrn.ADTEventTypeDSC=='Admission') & (adt_mrn.EffectiveDTS > min_admission), 'EffectiveDTS'].drop_duplicates().values
            admission = admissions[0]

            discharges = adt_mrn.loc[(adt_mrn.ADTEventTypeDSC=='Discharge') & (adt_mrn.EffectiveDTS > admission), 'EffectiveDTS'].drop_duplicates().values
            discharge = discharges[-1]

            cohort.loc[jloc, 'Admission'] = admission
            cohort.loc[jloc, 'Discharge'] = discharge

In [16]:
# edw_data = edw_data.loc[pd.notna(edw_data.MedicationTakenDTS)]
# edw_data = edw_data.loc[edw_data.MARActionDSC.apply(lambda x: x not in ['Canceled Entry', 'Stopped', 'Missed', 'Refused'])]

### Actual Medication Start.

In [17]:
edw_data = pd.read_csv(edw_data_path)
edw_data = preprocess_medication_data(edw_data)    
edw_data = remove_non_valid_entries(edw_data, verbose=False)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (14,16,17,20,23,24,26,27,28,31,32,36) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [18]:
# edw_data.loc[edw_data.MedicationDSC == 'isoflurane 99.9 % inhalation liquid', 'MRN'].unique().shape

In [19]:
all_meds = edw_data.MedicationDSC.str.lower().unique()

# # Covid 19 NCC Chou:
# meds_found = pd.DataFrame()
# meds_found['medname'] = meds_to_study
# meds_found['entries'] = np.nan

In [20]:
if project == 'covid_sedation':
    erx_med_pair = edw_data[['MedicationID', 'MedicationDSC']].copy()
    erx_med_pair.drop_duplicates(inplace=True)
    
    meds_to_study_df['MedicationDSC_Found'] = np.nan
    all_medicationdscs = []
    
    for jloc, row in meds_to_study_df.iterrows():
        match = erx_med_pair.loc[erx_med_pair.MedicationID == row.erx]
        if match.shape[0] > 0:
            meds_to_study_df.loc[jloc, 'MedicationDSC_Found'] = '; '.join(match.MedicationDSC.values)
            all_medicationdscs.extend(match.MedicationDSC.values)
            

In [21]:
def insert_medicationdsc_name_for_template_entries(df):

    med_ids_zzims = edw_data.loc[(edw_data.MedicationDSC =='ZZ IMS TEMPLATE')].MedicationID.unique()

    for med_id_tmp in med_ids_zzims:
        if med_id_tmp == 800001:
            continue

        names_tmp = list(edw_data.loc[edw_data.MedicationID == med_id_tmp].MedicationDSC.unique())
        names_tmp.remove('ZZ IMS TEMPLATE')
        if len(names_tmp) != 1:

            # see if it's the same type of Med for all, e..g. same med but different doses of med (tablets etc.):
            if len(np.unique([x.split(' ')[0]] for x in names_tmp)) == 1:
                names_tmp = [names_tmp[0].split(' ')[0]]

            else:
                print(f'Multiple Names for ERX {med_id_tmp}: {names_tmp}')
                continue

        edw_data.loc[edw_data.MedicationID == med_id_tmp, 'MedicationDSC'] = names_tmp[0]
        
    return df

In [22]:
if project == 'covid-risk-pred-ncc':
    for jloc, row in meds_found.iterrows():
        if 'placebo' in row.medname:
            medname = row.medname.replace(' ; placebo', '')
            entries_found = [x for x in all_meds if (('placebo' in x) | ('2020p' in x)) & (medname in x) & (not 'drop' in x)]
        else:
            medname = row.medname
            entries_found = [x for x in all_meds if (medname in x) & (not 'placebo' in x) & (not '2020p' in x) & (not 'drop' in x) & (not 'eye' in x)]
        meds_found.loc[jloc, 'entries'] = '; '.join(entries_found)
    meds_found['medname'] = meds_found['medname'].apply(lambda x: x.replace(' ; placebo', '__placebo'))
    meds_found.to_csv('NCC_Study_Meds_Founds.csv', index=False)
    
    ncc_study = pd.DataFrame()
    ncc_study['MRN'] = mrns
    ncc_study['Meds_Given_Generic'] = ''
    ncc_study['Meds_Given_Specific'] = ''
    
    for jloc, row in ncc_study.iterrows():
        mrn = row.MRN
        data_mrn = edw_data[edw_data.MRN == mrn]

        for kloc, krow in meds_found.iterrows():
            entries = krow.entries
            if pd.isna(entries): continue
            entries = entries.split('; ')

            entries_found = [x for x in entries if x in data_mrn.MedicationDSC.str.lower().values]
            if len(entries_found) > 0:
                ncc_study.loc[jloc, 'Meds_Given_Specific'] += '; '.join(entries_found) + '; '
                ncc_study.loc[jloc, 'Meds_Given_Generic'] += krow.medname +'; '

                
    ncc_study['Meds_Given_Generic'] = ncc_study['Meds_Given_Generic'].apply(lambda x: x[:-2] if x[-2:] == '; ' else x)
    ncc_study['Meds_Given_Specific'] = ncc_study['Meds_Given_Specific'].apply(lambda x: x[:-2] if x[-2:] == '; ' else x)
    
    ncc_study.to_csv('NCC_Study_Meds_Given.csv', index=False)

In [23]:
# df = edw_data.copy()

In [24]:
# edw_data = remove_non_valid_entries(edw_data)

### remove all Medication data that is outside of hospital stay of interest: (as in Admission and Discharge in cohort)

In [26]:
for jloc, row in cohort.iterrows():

    mrn_tmp = row.mrn
    
    index_outside_of_stay_of_interest = edw_data.loc[(edw_data.MRN==mrn_tmp) & ((edw_data.MedicationTakenDTS<row.Admission) | (edw_data.MedicationTakenDTS>row.Discharge))].index
    edw_data.drop(index_outside_of_stay_of_interest, inplace=True)

    index_outside_of_stay_of_interest = edw_data.loc[(edw_data.MRN==mrn_tmp) & ((edw_data.OrderEndDTS<row.Admission) | (edw_data.OrderStartDTS>row.Discharge))].index
    edw_data.drop(index_outside_of_stay_of_interest, inplace=True)    

### remove all Medication data that is not in the medication list

In [30]:
# if med_list_type == 'name_erx'
# elif med_list_type == 'name':
found_med_bool = edw_data.MedicationDSC.apply(lambda x: any([x_med.lower() in x.lower() for x_med in meds_to_study]))
edw_data = edw_data.loc[found_med_bool]

In [33]:
def determine_IsInfusion(df):
    '''
    IsInfusion is True if DiscreteFrequencyDSC says 'Continuous' or if, for a whole order, DiscreteDoseAMT is empty but InfusionRateNBR is available.
    '''
    
    df['IsInfusion'] = (df.DiscreteFrequencyDSC=='Continuous PRN') | (df.DiscreteFrequencyDSC=='Continuous')
    
    for order_id in pd.unique(df.OrderID):
        df_order = df[df.OrderID == order_id]
        if (np.all(pd.isna(df_order.DiscreteDoseAMT)) & (~np.any(pd.isna(df_order.InfusionRateNBR)))):
            index_order = df_order.index
            df.loc[index_order, 'IsInfusion'] = True
    
    return df


def determine_dose_and_unit(df):
    '''
    Aim: Here, to the EDW output, three columns are appended: IsInfusion, Dose, Unit
    IsInfusion: boolean indicator if entry belongs to continuous infusion.
    Dose: Dose (in mg or mg/h)
    Unit: colum with 'mg' or 'mg/h'
    
    Input: dataframe, raw EDW medication output, after running preprocess_medication_data().
    Output: dataframe, raw EDW medication output + 3 additional columns, see above.
    
    This function therefore contains all the logic to determine and convert the original doses and units that are found in various fields, depending also on the type of medication.
    The output as additional columns will make further processing easy and human-read-friendly.
    '''
    
    df['IsInfusion'] = np.nan
    df['Unit'] = np.nan
    df['Dose'] = np.nan
    
    df = determine_IsInfusion(df)
    
    

In [34]:
valid_units = ['mg', 'mg/hr', 'g', 'g/hr', 'mcg', 'mcg/hr', 'mg/min', 'mcg/min', 'mcg/hr', ]
weight_dependent_units = ['mcg/kg/min', 'mcg/kg/hr', 'mcg/kg', 'mg/kg', 'g/kg']

In [35]:
def valid_unit_and_order_dose(df):
    
   # Check for good DoseUnitDSC and SigTXT_Order combination:
    valid_dose_order_and_unit = df.loc[(pd.isna(df.Unit)) & \
                                       (df.DoseUnitDSC.apply(lambda x: x in valid_units)) & \
                                       (pd.notna(df.SigTXT_Order)) & \
                                       (df.SigTXT_Order.apply(lambda x: type(x) in [int, float]))].index
    df.loc[valid_dose_order_and_unit, 'Unit'] = df.loc[valid_dose_order_and_unit, 'DoseUnitDSC'] 
    df.loc[valid_dose_order_and_unit, 'Dose'] = df.loc[valid_dose_order_and_unit, 'SigTXT_Order']  
    
    return df


def valid_unit_and_discrete_dose(df):
    
    # Check for good DoseUnitDSC and DiscreteDoseAMT combination:
    valid_discrete_dose_and_unit = df.loc[(pd.isna(df.Unit)) & \
                                           (df.DoseUnitDSC.apply(lambda x: x in valid_units)) & \
                                           (pd.notna(df.DiscreteDoseAMT)) & \
                                           (df.DiscreteDoseAMT.apply(lambda x: '-' not in str(x)))].index
    df.loc[valid_discrete_dose_and_unit, 'Unit'] = df.loc[valid_discrete_dose_and_unit, 'DoseUnitDSC']
    df.loc[valid_discrete_dose_and_unit, 'Dose'] = df.loc[valid_discrete_dose_and_unit, 'DiscreteDoseAMT'].astype(float)
    
    return df

def icu_sleep_studydrug(df):

    # Handle the ICU-Sleep Placebo/Study Drug, as it behave slightly different than other entries. We keep the original Infusion Rate and InfusionRate Unit, as we are blinded.
    icu_sleep_study_drug = df.loc[df.MedicationDSC.apply(lambda x: '2017P000090'.lower() in x.lower())].index
    df.loc[icu_sleep_study_drug, 'Unit'] = df.loc[icu_sleep_study_drug, 'InfusionRateUnitDSC']
    df.loc[icu_sleep_study_drug, 'Dose'] = df.loc[icu_sleep_study_drug, 'InfusionRateNBR']
    
    return df


def valid_infusion_rate_and_dose_from_med_name(df):
    
    # Entries with valid InfusionRate (mL/hr) and extraction of dose from Medicationname
    ml_h_infusion = df.loc[(pd.isna(df.Unit)) & \
                              (df.InfusionRateUnitDSC.apply(lambda x: str(x).lower() == 'ml/hr'))].index
    no_dose_found = []
    for jloc, row in df.loc[ml_h_infusion].iterrows():
        unit_dose = re.search(r"\d+ .{1,3}/ml?", row.MedicationDSC)
        unit_dose_mg_ml = re.search('\d+ mg/\d+ ml' , row.MedicationDSC)
        
        if unit_dose is not None:
            unit_dose = unit_dose[0]
            # make sure units match (e.g. /ml from MedicationDSC and ml/ from Infusionrate)
            dose = np.float(unit_dose.split(' ')[0]) * np.float(row.InfusionRateNBR)
            unit = unit_dose.split(' ')[1]
            assert unit.split('/')[1].lower() == row.InfusionRateUnitDSC.split('/')[0].lower()
            assert '/ml' in unit
            unit = unit.split('/')[0] + '/' + row.InfusionRateUnitDSC.split('/')[1].lower()
        
        elif unit_dose_mg_ml is not None:
            mg_ml = unit_dose_mg_ml[0]
            mg = np.float(mg_ml.split('/')[0].replace(' mg', ''))
            ml = np.float(mg_ml.split('/')[1].replace(' ml', ''))
            dose = mg/ml * np.float(row.InfusionRateNBR)
            
            assert row.InfusionRateUnitDSC.split('/')[0].lower() == 'ml'
            
            unit = 'mg/' + row.InfusionRateUnitDSC.split('/')[1].lower()

        else:
            if not row.MedicationDSC in no_dose_found:
                no_dose_found.append(row.MedicationDSC)
            continue

        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit
 
    if len(no_dose_found) > 0:
        print(f'Dose Parsing from MedicationDSC failed for {no_dose_found}.')
    
    return df


def infusions_sigtxt_and_med_name(df):
    
    # Entries with valid units in medication name and info about dose in SigTxt, verified for a few examples in Epic.
    candidates = df.loc[(pd.isna(df.Unit)) & \
                        (pd.isna(df.InfusionRateUnitDSC)) & \
                        (pd.isna(df.DiscreteDoseAMT)) & \
                        (df.MedicationDSC.apply(lambda x: 'ml' in str(x).lower() ))].index
    no_dose_found = []
    for jloc, row in df.loc[candidates].iterrows():
        unit_dose = re.search(r".{1,3}/ml?", row.MedicationDSC)
        unit_dose_mg_ml = re.search('\d+ mg/\d+ ml' , row.MedicationDSC)
        
        if unit_dose is not None:
            unit = unit_dose[0].replace('/ml', '').replace(' ', '')
            # make sure units match (e.g. /ml from MedicationDSC and ml/ from Infusionrate)
            dose = np.float(row.SigTXT_Order)

        elif unit_dose_mg_ml is not None:
            mg_ml = unit_dose_mg_ml[0]
            unit = 'mg'
            dose = np.float(row.SigTXT_Order)
            
        else:
            continue

        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit
 
    return df

def self_administered_meds(df, verbose=False):

    meds_self_controlled = [x for x in pd.unique(df.MedicationDSC) if ('PCA'.lower() in x) | ('PCEA'.lower() in x)]
    if verbose:
        print(f'For the following meds, patient-self-administration is assumed because of "PCA" or "PCEA" in Med Name. Placeholder dose of 0.01 is imputed.')
        print(meds_self_controlled)
    
    for med_self_controlled in meds_self_controlled:
        
        loc_self_controlled = df.loc[(pd.isna(df.Unit)) & \
                                    (df.MedicationDSC == med_self_controlled)].index
        loc_self_controlled_stopped = df.loc[(pd.isna(df.Unit)) & \
                                    (df.MedicationDSC == med_self_controlled) & \
                                    (df.MARActionDSC == 'Stopped')].index
        
                                    
        df.loc[loc_self_controlled, 'Unit'] = 'mg/hr'
        df.loc[loc_self_controlled, 'Dose'] = 0.01  
        df.loc[loc_self_controlled_stopped, 'Dose'] = 0
        
    return df


def patches(df):
    
    loc_patches_0 = df.loc[(pd.isna(df.Unit)) & \
                            (df.DoseUnitDSC=='patch') & \
                           (df.MARActionDSC.apply(lambda x: x in ['Patch Removed', 'Due']))].index
    df.loc[loc_patches_0, 'Dose'] = 0
    df.loc[loc_patches_0, 'Unit'] = 'mcg/hr'
    
    loc_patches_1 = df.loc[(pd.isna(df.Unit)) & \
                            (df.DoseUnitDSC=='patch') & \
                           (df.MARActionDSC.apply(lambda x: x in ['Patch Applied']))].index  
    
    for jloc, row in df.loc[loc_patches_1].iterrows():
        
        unit_dose_patch = re.search(r"(\d+[.])?\d+ .+? ", row.MedicationDSC)[0]
        dose = np.float(unit_dose_patch.split(' ')[0])
        unit = unit_dose_patch.split(' ')[1]
        if '/24' in unit: # that means descriptions says m(c)g/24 hours.
            dose = dose/24
            unit = unit.replace('24', 'hr')            

        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit

    return df


def tablets(df):
    '''Many tablets might be already handled by DiscreteDose+Unit processing script, however
    many entries seem to have no DiscreteDoseEntry, but Unit information is clear from Medication Name
    and dose can be extracted from SigTXT_Order'''
    
    loc_tablets = df.loc[(pd.isna(df.Unit)) & \
                           (df.MedicationDSC.apply(lambda x: np.any(['tablet' in x, 'capsule' in x]))) & \
                        (pd.notna(df.SigTXT_Order))].index
    
    for jloc, row in df.loc[loc_tablets].iterrows():
        
        unit = [x for x in [' mg ', ' g '] if x in row.MedicationDSC]
        assert len(unit) == 1, f'No or multiple Units found for tablet {row.MedicationDSC}.'
        
        unit = unit[0].replace(' ', '')
        try:
            dose = np.float(row.SigTXT_Order)
        except:
            continue
       
        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit
        
    # Similar, except Info is in DiscreteDose expect of SigTXT_ORder
    loc_tablets = df.loc[(pd.isna(df.Unit)) & \
                           (df.MedicationDSC.apply(lambda x: np.any(['tablet' in x, 'capsule' in x]))) & \
                        (pd.notna(df.DiscreteDoseAMT))].index
    
    for jloc, row in df.loc[loc_tablets].iterrows():
        
        unit = [x for x in [' mg ', ' g '] if x in row.MedicationDSC]
        assert len(unit) == 1, f'No or multiple Units found for tablet {row.MedicationDSC}.'
        
        unit = unit[0].replace(' ', '')
        try:
            dose = np.float(row.DiscreteDoseAMT)
        except:
            continue
            
        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit

    return df

def impute_missing_dose_unit(df, med_unit_dict):

    df['DoseUnitDSC_Imputed'] = 0
        
    for jloc, row in df.loc[pd.isna(df.Unit)].iterrows():

        unit = med_unit_dict[row.MedicationID]
        if len(unit) == 1:
            df.loc[jloc, 'DoseUnitDSC'] = unit[0]
            df.loc[jloc, 'DoseUnitDSC_Imputed'] = 1
            
    return df


def weight_dependent_dose(df):

    weight_dependent_dose = df.loc[(pd.isna(df.Unit)) & \
          (df.DoseUnitDSC.apply(lambda x: '/kg' in str(x)))].index

    for jloc, row in df.loc[weight_dependent_dose].iterrows():
        dose = None
        if pd.notna(row.SigTXT_Order):
            dose = np.float(row.SigTXT_Order)
        elif pd.notna(row.DiscreteDoseAMT):
            dose = np.float(row.DiscreteDoseAMT)
        if not dose is None:
            df.loc[jloc, 'Unit'] = row.DoseUnitDSC
            df.loc[jloc, 'Dose'] = dose
            
    return df


def use_mean_of_dose_range(df):

    dose_range_available = df.loc[pd.isna(df.Unit) & \
                (df.DiscreteDoseAMT.apply(lambda x: '-' in str(x))) & \
               (pd.isna(df.SigTXT_Order)) & \
               (pd.notna(df.DoseUnitDSC))].index

    df.loc[dose_range_available, 'Unit'] = df.loc[dose_range_available, 'DoseUnitDSC']
    df.loc[dose_range_available, 'Dose'] = df.loc[dose_range_available, 'DiscreteDoseAMT'].str.split('-').apply(lambda x: np.mean([float(y) for y in x]))

    return df


def sodium_chloride_iv(df):
    
    # 'sodium chloride 0.9 % intravenous solution'
    df.loc[(df.MedicationID == 27838), 'Dose'] = 0.09 * df.loc[(df.MedicationID == 27838), 'InfusionRateNBR']
    df.loc[(df.MedicationID == 27838), 'Unit'] = 'mg/hr'
    
    # 'sodium chloride 0.9 % bolus:
    df.loc[(df.MedicationID == 400291), 'Dose'] = 0.09 * df.loc[(df.MedicationID == 400291), 'SigTXT_Order']
    df.loc[(df.MedicationID == 400291), 'Unit'] = 'mg'
        
    # 'sodium chloride 0.45 % intravenous solution'
    df.loc[(df.MedicationID == 7318), 'Dose'] = 0.045 * df.loc[(df.MedicationID == 7318), 'InfusionRateNBR']
    df.loc[(df.MedicationID == 7318), 'Unit'] = 'mg/hr'
    
    # 'sodium chloride 0.45 % bolus:
    df.loc[(df.MedicationID == 400292), 'Dose'] = 0.045 * df.loc[(df.MedicationID == 400292), 'SigTXT_Order']
    df.loc[(df.MedicationID == 400292), 'Unit'] = 'mg'    
    
    return df

def set_isoflurane_to_dummy(df):
    '''Isoflurane is not well captured in the system. We can only make a binary statement, therefore fill values with dummy value 1.'''
    
    df.loc[(df.MedicationID == 127796), 'Dose'] = 1
    df.loc[(df.MedicationID == 127796), 'Unit'] = 'mg'
    
    return df


def init_medname_dict(df_mrn):
    print(df_mrn.columns)
    medname_original_new = {}
    for med in df_mrn[medname_col].unique():

        if '2017P000090'.lower() in med:
            new_name = 'DEXMEDETOMIDINE_OR_PLACEBO_(2017P000090)'.lower()
        else:
            new_name = med.replace(' ', '_').replace('-','_').replace('%', 'perc').replace('/', 'per').replace(',','_').replace(';', '_')

        medname_original_new[med] = new_name
        
    return medname_original_new

In [36]:
def init_med_unit_dict(df, verbose=False):


    med_unit_dict = {}

    for med in df.MedicationID.unique():

        # check if for this med, only one unique unit is used for all patients:
        unique_unit = None
        units_for_this_med = pd.unique(df[df.MedicationID == med].DoseUnitDSC.dropna())
        med_unit_dict[med] = units_for_this_med
        
        if verbose:
            if len(units_for_this_med) == 0:
                print(f'0: {med}')
            elif len(units_for_this_med) > 1:
                print(f'>1: {med}')
                
    return med_unit_dict


In [42]:
def medication_processing_routine(df, remove_original_unit_dose_cols=True, verbose=False):
    '''
    Run the whole medication processing routine. This function contains all the logic how the doses and units are derived from the EDW fields.
    Input:
    df = edw medication data output after preprocess_medication_data() and remove_non_valid_entries()
    Output: 2 dataframes, similar to input but with 3 additional columns: IsInfusion, Dose, Unit
    medications: all entries that could succesfully handeled with the logic implemented
    medications_nonsuccessful: all entries that where a Dose or Unit could not be found (is usually <0.1% of the cases, when clearly sth. is missing in the EDW fields...)
    '''

    df['IsInfusion'] = np.nan
    df['Unit'] = np.nan
    df['Dose'] = np.nan

    med_unit_dict = init_med_unit_dict(df)
    df = determine_IsInfusion(df)

    print('Manual Correction for Rocuronium entries - set dose to mg. [Done for Covid Delirium Project]:')
    df.loc[(df.MedicationID == 95811) & pd.notna(df.SigTXT_Order) & pd.isna(df.DoseUnitDSC), 'DoseUnitDSC'] = 'mg'
    
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('valid_unit_and_order_dose...')
    df =  valid_unit_and_order_dose(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('valid_unit_and_discrete_dose...')
    df = valid_unit_and_discrete_dose(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('icu_sleep_studydrug...')
    df =  icu_sleep_studydrug(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('valid_infusion_rate_and_dose_from_med_name...')
    df = valid_infusion_rate_and_dose_from_med_name(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        ### up to here, usually >95% of the medications can be successfully processed.
        print('self_administered_meds...')
    df = self_administered_meds(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        ### up to here, usually > 98% of meds are processed.
        print('patches...')
    df = patches(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('tablets...')
    df = tablets(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        ### up to here, usually > 99% of meds are processed.
        print('impute missing dose units...')
        # impute missing dose units and run above code again
    df = impute_missing_dose_unit(df, med_unit_dict)
    df = valid_unit_and_order_dose(df)
    df = valid_unit_and_discrete_dose(df)
    df = icu_sleep_studydrug(df)
    df = valid_infusion_rate_and_dose_from_med_name(df)
    df = self_administered_meds(df)
    df = patches(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('use_mean_of_dose_range')
    df = use_mean_of_dose_range(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('weight dependent doses...')
    df = weight_dependent_dose(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('sodium chloride ivs...')
                
    df = sodium_chloride_iv(df)
    df = set_isoflurane_to_dummy(df)
    
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)

    if remove_original_unit_dose_cols:
        # we have standardized the dose/unit here, so we can remove all EDW dose/unit fields:
        cols_remove = ['DoseUnitCD', 'DoseUnitDSC', 'MinimumDoseAMT', 'MaximumDoseAMT', 'InfusionRateNBR', 'InfusionRateUnitCD', 'InfusionRateUnitDSC', 'DefinedDailyDoseNBR', 'DiscreteDoseAMT', 'DiscreteDispenseUnitDSC']
        cols_remove = [x for x in cols_remove if x in df.columns]
        df.drop(cols_remove, axis = 1, inplace = True)

    df.drop(['DoseUnitDSC_Imputed'], axis = 1, inplace = True)
    
    df['MedicationDSCNew'] = np.nan
    medname_original_new = init_medname_dict(df)
    for original_medname in df.MedicationDSC.unique():
        df.loc[df.MedicationDSC == original_medname, 'MedicationDSCNew'] = medname_original_new[original_medname]

    medications_nonsuccessful = df.loc[pd.isna(df.Unit)]
    medications = df.loc[pd.notna(df.Unit)]

    if verbose:
        print(f'Number of entries that could be successfully processed: {medications.shape[0]}')
        print(f'Number of entries that could not be successfully processed: {medications_nonsuccessful.shape[0]}')
        print(f'All MedicationDSCs that could not be succesfully processed:\n{medications_nonsuccessful.MedicationDSC.value_counts()}')
        
    return medications, medications_nonsuccessful

In [43]:
# time series:

if project in ['sample', 'icu-sleep', 'covid_delirium']:
    medname_col = 'MedicationDSC'

elif project == 'covid_sedation':
    medname_col = 'MedicationGenericName'


In [39]:
edw_data_copy = edw_data.copy()

In [40]:
edw_data = edw_data_copy.copy()

In [44]:
meds, meds_nonsuccess = medication_processing_routine(edw_data, remove_original_unit_dose_cols=True, verbose=True)

Manual Correction for Rocuronium entries - set dose to mg. [Done for Covid Delirium Project]:
Dose Parsing from MedicationDSC failed for ['phenobarbital ivpb loading dose (100 ml)', 'isoflurane 99.9 % inhalation liquid'].
Dose Parsing from MedicationDSC failed for ['phenobarbital ivpb loading dose (100 ml)', 'isoflurane 99.9 % inhalation liquid'].
Index(['OrderID', 'MRN', 'PatientEncounterID', 'MedicationID', 'MedicationDSC',
       'OrderInstantDTS', 'OrderStartDTS', 'OrderEndDTS', 'MedicationTakenDTS',
       'MARActionCD', 'MARActionDSC', 'RouteDSC', 'RouteCD', 'DurationNBR',
       'DurationUnitCD', 'DurationUnitDSC', 'ActionSourceDSC', 'SigTXT_Order',
       'SigTXT_Administered', 'MedicationDiscontinueReasonDSC',
       'DiscreteFrequencyDSC', 'IsInfusion', 'Unit', 'Dose',
       'MedicationDSCNew'],
      dtype='object')


KeyError: 'MedicationGenericName'

In [ ]:
print(f'meds success: {meds.shape[0]}')
print(f'meds non-success: {meds_nonsuccess.shape[0]}') # 51

In [ ]:
meds_nonsuccess

In [38]:
if 1:
    meds.to_csv(f'medications_{project}_successfully_processed.csv', index = False)

In [39]:
if 1:
    meds_nonsuccess.to_csv(f'medications_{project}_not_successfully_processed.csv', index = False)

In [5]:
if 1:
    meds = pd.read_csv(f'medications_{project}_successfully_processed.csv')

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (22) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [13]:
# meds.loc[np.isin(meds.MedicationID, [27908, 41382, 27905]), :].MRN.unique()

In [40]:
meds = meds.copy()
meds['MedicationGenericName'] = np.nan

In [41]:
med_tmp = 'valproic_acid_(as_sodium_salt)_250_mgper5_ml_(5_ml)_oral_solution_transdermal'
# generic_name = [x for x in meds_to_study if x in med_tmp.replace('_', ' ')]
meds.loc[meds.MedicationDSC == med_tmp]

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,IsInfusion,Unit,Dose,MedicationDSCNew,MedicationGenericName


In [42]:
# generic_name

In [43]:
# meds_to_study

In [44]:
if med_list_type == 'name_erx':
    meds = meds.loc[np.isin(meds.MedicationID, meds_to_study_df.erx.values)].copy()

    for erx_tmp, name_tmp in zip(meds_to_study_df.erx.values, meds_to_study_df.med_name.values):
        meds.loc[meds.MedicationID == erx_tmp, 'MedicationGenericName'] = name_tmp 
        
elif med_list_type == 'name':
    meds_contained = meds.MedicationDSC.unique()
    for med_tmp in meds_contained:
        generic_name = [x for x in meds_to_study if x in med_tmp]
        if (len(generic_name) == 0) & ('_' in med_tmp):
            generic_name = [x for x in meds_to_study if x in med_tmp.replace('_', ' ')]
        if len(generic_name) == 0:
            print(f'No generic medication categories found for {med}: {generic_name}. Skip.')
            continue
        if len(generic_name)>1:    
            # first, remove medication name that are a substring of another name:
            for x in generic_name:
                if not x in generic_name: 
                    continue #this is werid but has to be done because .remove() is inplace operation.
                if any([x in y for y in set(generic_name) - set([x])]):
                    generic_name.remove(x)
        if len(generic_name)>1:            
            print(f'Multiple generic medication categories found for {med}: {generic_name}. Take the first.')
        generic_name = generic_name[0]

        meds.loc[meds.MedicationDSC == med_tmp,'MedicationGenericName'] = generic_name


In [45]:
# convert to mg or mg/h

# convert equivalency.

In [46]:
def convert_units(dose, unit, body_weight_kg=None):
        # normalize the unit to mg (bolus) or mg/hr (infusion)
    unit = unit.lower()
    
    if unit in ['mg', 'mg/hr']:
        pass
    elif unit in ['g', 'g/hr']:
        dose = dose * 1000.
    elif unit in ['mcg', 'mcg/hr']:
        dose = dose / 1000.
    elif unit == 'mg/min':
        dose = dose * 60.
    elif unit == 'mcg/min':
        dose = dose * 60. / 1000.
    elif unit == 'mcg/hr':
        dose = dose / 1000.
    elif unit == 'mcg/h':
        dose = dose / 1000.
    elif unit == 'mcg/kg/min':
        dose = dose * 60. / 1000.* body_weight_kg
    elif unit == 'mcg/kg/hr':
        dose = dose / 1000.* body_weight_kg
    elif unit == 'mcg/kg':
        dose = dose / 1000.* body_weight_kg
    elif unit == 'mg/kg':
        dose = dose * body_weight_kg
    elif unit == 'g/kg':
        dose = dose * 1000.* body_weight_kg    
    elif unit == 'ml':
        # this is for 'infiltration' and 'nebulization'. does not happen often. this conversino just serves as an estimate (w/o density), results of those meds should be interpreted accordingly.
        dose = dose * 1000
    else:
        raise ValueError('Unknown unit: %s'%(unit))
        
    return dose

In [47]:
meds.Unit.unique()

array(['mcg/kg/min', 'mg', 'mg/hr', 'mg/kg', 'mcg/hr', 'mcg', 'mcg/kg',
       'mcg/kg/hr'], dtype=object)

##### 

In [48]:
def correct_weight_dependent_unit_with_flowsheet_weight(meds, cohort):
    
    # some irregular entries might still have weight_dependent units, fix them with weight obtained from flowsheets:

    df_weight_dependent = meds.loc[meds.Unit.apply(lambda x: '/kg' in x)]
    if df_weight_dependent.shape[0] > 0:
        for jloc, row in df_weight_dependent.iterrows():
            try:
                weight = cohort.loc[cohort.PatientID == row.PatientID, 'Weight'].values[0]
            except:
                weight = cohort.loc[cohort.mrn == row.MRN, 'Weight'].values[0]
            
            meds.loc[jloc, 'Dose'] = meds.loc[jloc, 'Dose'] * weight
            meds.loc[jloc, 'Unit'] = meds.loc[jloc, 'Unit'].replace('/kg', '')

    return meds


def convert_to_mg_h(meds):
    
    unit_conversion_to_mg = {
    'mcg': [0.001, 'mg'],
    'mg': [1, 'mg'],
    'g' : [1000, 'mg'],
    
    'mcg/hr': [0.001, 'mg/hr'],
    'mg/hr': [1, 'mg/hr'],
    'g/hr': [1000, 'mg/hr'],
    
    'mcg/min': [0.001 * 60, 'mg/hr'],
    'mg/min': [1 * 60, 'mg/hr'],
    'g/min': [1000 * 60, 'mg/hr']
    }

    for unit in unit_conversion_to_mg:
        meds.loc[meds.Unit == unit, 'Dose'] = meds.loc[meds.Unit == unit, 'Dose'] * unit_conversion_to_mg[unit][0]
        meds.loc[meds.Unit == unit, 'Unit'] = unit_conversion_to_mg[unit][1]
        
    return meds

In [49]:
meds = correct_weight_dependent_unit_with_flowsheet_weight(meds, cohort)
meds = convert_to_mg_h(meds)

In [50]:
print(meds.Unit.unique())

['mg/hr' 'mg']


In [51]:
if 1:
    meds.loc[meds.Unit == ' mg/ml', 'Unit'] = 'mg'
    meds.loc[meds.Unit == 'mcg/ml', 'Unit'] = 'mcg'
    meds = correct_weight_dependent_unit_with_flowsheet_weight(meds, cohort)
    meds = convert_to_mg_h(meds)

In [52]:
# Let's convert Opioids, Benzos, and Antipsychotics to Equivalents:

def general_medication_dose_equivalencies():

    # define equivalent doses (scaler is inverse of those values): 
    opioids_ratios = {
        'morphine_parenteral': 1, 
        'morphine_oral': 3,
        'codeine_oral': 20,
        'hydromorphone_parenteral': 0.15,
        'hydromorphone_oral': 0.75,
        'oxycodone_oral': 2,
        'hydrocodone_oral': 3,
        'oxymorphone_parenteral': 0.1,
        'oxymorphone_oral': 0.15,
        'levorphanol_parenteral': 0.2,
        'levorphanol_oral': 0.4,
        'methadone_parenteral': 1,
        'methadone_oral': 2,
        'fentanyl_parenteral': 0.01,
        'fentanyl_transdermal': 7.2,
        'fentanyl_oral': 0.01,
        'buphrenorphine_parenteral': 0.03,
        'buphrenorphine_transdermal': 0.001,
        'tapentadol': 0.75,
        'tramadol_oral': 10,
        'meperidine_oral': 30,
        'meperidine_parenteral': 30,
        'pentazocine_oral': 18,
        'propoxyphene_oral': 6.5,
        'nalbuphine_oral': 0.25,
        'remifentanil_parenteral': 0.01,
        'sufentanil_parenteral': 0.001,
    }
    

    # define equivalent doses (scaler is inverse of those values): 
    benzos_ratios = {
        'alprazolam_oral': 0.5, 
        'chlordiazepoxide_oral': 10, 
        'clonazepam_oral': 0.25, 
        'diazepam_oral': 5, 
        'diazepam_parenteral': 5, 
        'lorazepam_oral': 1, 
        'lorazepam_parenteral': 1, 
        'midazolam_oral': 2, 
        'midazolam_parenteral': 4, 
        'oxazepam_oral': 10, 
        'phenobarbital_oral': 15, 
        'propofol_parenteral': 160,
        'dexmedetomidine_parenteral': 59.3
    }

    antipsychotics_ratios = {
        'chlorpromazine_oral': 300, 
        'chlorpromazine_parenteral': 100, 
        'fluphenazine_oral': 10,
        'fluphenazine_parenteral': None, 
        'haloperidol_oral': 8 , 
        'haloperidol_parenteral': 8, 
        'loxapine_oral': 100, 
        'loxapine_parenteral': None, 
        'perphenazine_oral': 30, 
        'perphenazine_parenteral': 10, 
        'pimozide_oral': 4, 
        'pimozide_parenteral': None, 
        'prochlorperazine_oral': 100, 
        'prochlorperazine_parenteral': 50, 
        'trifluoperazine_oral': 20, 
        'trifluoperazine_parenteral': 8, 
        'thioridazine_oral': 300, 
        'thioridazine_parenteral': 0, 
        'thiothixene_oral': 30, 
        'thiothixene_parenteral': None, 
        'tiothixene_oral': 30, 
        'tiothixene_parenteral': None, 
        'aripiprazole_oral': 15, 
        'aripiprazole_parenteral': 15, 
        'asenapine_oral': 20, 
        'asenapine_parenteral': None, 
        'brexpiprazole_oral': None, 
        'brexpiprazole_parenteral': None, 
        'cariprazine_oral': None,
        'cariprazine_parenteral': None,
        'clozapine_oral': 300, 
        'clozapine_parenteral': 300, 
        'iloperidone_oral': None, 
        'iloperidone_parenteral': None, 
        'lurasidone_oral': 60, 
        'lurasidone_parenteral': None, 
        'olanzapine_oral': 10, 
        'olanzapine_parenteral': 10, 
        'paliperidone_oral': 6, 
        'paliperidone_parenteral': 2.5, 
        'quetiapine_oral': 400, 
        'quetiapine_parenteral': None, 
        'risperidone_oral': 5, 
        'risperidone_parenteral': 2.7, 
        'ziprasidone_oral': 80, 
        'ziprasidone_parenteral': 40, 
    }
    
    return opioids_ratios, benzos_ratios, antipsychotics_ratios


def specific_medication_dose_equivalencies():
    # OPIOID DICTIONARIES:
    opioids_meds_category = {}
    opioids_meds_family = {}
    opioids_meds_ratios = {}

    meds_all = np.concatenate([meds.MedicationDSC.unique(), (meds.MedicationDSCNew.unique())])
    for med in opioids_meds:
        opioid_dsc = [x for x in meds_all if med in x]
        for med_tmp in opioid_dsc:
            if any(['injection' in med_tmp, 'intravenous' in med_tmp, 'infusion' in med_tmp, 
                    'pcea'in med_tmp, 'pca' in med_tmp, 'bag' in med_tmp, 'bolus' in med_tmp, 'ivpb' in med_tmp]):
                cat = 'parenteral'
            elif any(['transdermal' in med_tmp, 'patch' in med_tmp]):
                cat = 'transdermal'
            elif any(['oral', 'tablet']):
                cat = 'oral'
            else:
                print(f'No category found for {med}!')

            opioids_meds_category[med_tmp] = cat
            opioids_meds_family[med_tmp] = med

            if med + '_' + cat in opioids_ratios:
                opioids_meds_ratios[med_tmp] = opioids_ratios[med + '_' + cat]
            elif med in morphine_ratios:
                opioids_meds_ratios[med_tmp] = opioids_ratios[med]
            else:
                print(f'{med} is not found in morphine_ratios!')


    # BENZOS DICTIONARIES:
    benzos_meds_category = {}
    benzos_meds_family = {}
    benzos_meds_ratios = {}

    for med in benzos_meds:
#         print(f'\n{med}:')
        benzos_dsc = [x for x in meds_all if med in x]
#         print(benzos_dsc)
        for med_tmp in benzos_dsc:
            if any(['injection' in med_tmp, 'intravenous' in med_tmp, 'infusion' in med_tmp, 
                    'pcea'in med_tmp, 'pca' in med_tmp, 'bag' in med_tmp, 'bolus' in med_tmp, 'ivpb' in med_tmp]):
                cat = 'parenteral'
            elif any(['transdermal' in med_tmp, 'patch' in med_tmp]):
                cat = 'transdermal'
            elif any(['oral', 'tablet']):
                cat = 'oral'
            else:
                print(f'No category found for {med}!')

            benzos_meds_category[med_tmp] = cat
            benzos_meds_family[med_tmp] = med

            if med + '_' + cat in benzos_ratios:
                benzos_meds_ratios[med_tmp] = benzos_ratios[med + '_' + cat]
            elif med in benzos_ratios:
                benzos_meds_ratios[med_tmp] = benzos_ratios[med]
            else:
                print(f'{med} is not found in benzos_ratios!')


    # ANTIPSYCH DICTIONARIES:
    antipsychotics_meds_category = {}
    antipsychotics_meds_family = {}
    antipsychotics_meds_ratios = {}

    for med in antipsychotics_meds:
    #     print(f'\n{med}:')
        antipsychotics_dsc = [x for x in meds_all if med in x]
        for med_tmp in antipsychotics_dsc:
            if any(['injection' in med_tmp, 'intravenous' in med_tmp, 'infusion' in med_tmp, 
                    'pcea'in med_tmp, 'pca' in med_tmp, 'bag' in med_tmp, 'bolus' in med_tmp, 'ivpb' in med_tmp]):
                cat = 'parenteral'
            elif any(['transdermal' in med_tmp, 'patch' in med_tmp]):
                cat = 'transdermal'
            elif any(['oral', 'tablet']):
                cat = 'oral'
            else:
                print(f'No category found for {med}!')

            antipsychotics_meds_category[med_tmp] = cat
            antipsychotics_meds_family[med_tmp] = med

            if med + '_' + cat in antipsychotics_ratios: 
                antipsychotics_meds_ratios[med_tmp] = antipsychotics_ratios[med + '_' + cat]
            elif med in antipsychotics_ratios:
                antipsychotics_meds_ratios[med_tmp] = antipsychotics_ratios[med]
            else:
                print(f'{med} {cat} is not found in antipsychotics_ratios!')
                print(med_tmp)
                
    return opioids_meds_ratios, benzos_meds_ratios, antipsychotics_meds_ratios

def compute_equivalencies(meds):

    meds['Dose_Opioid_Equivalent'] = np.nan
    meds['Dose_Benzo_Equivalent'] = np.nan
    meds['Dose_Antipsychotics_Equivalent'] = np.nan
    equivalency_ratios_list = [opioids_meds_ratios, benzos_meds_ratios, antipsychotics_meds_ratios]
    equivalency_names = ['Dose_Opioid_Equivalent', 'Dose_Benzo_Equivalent', 'Dose_Antipsychotics_Equivalent']

    for equi_classname, equi_ratio in list(zip(equivalency_names, equivalency_ratios_list)):
        for equi_med in equi_ratio:
            meds.loc[meds.MedicationDSC == equi_med, equi_classname] = meds.loc[meds.MedicationDSC == equi_med, 'Dose'] * equi_ratio[equi_med]

    return meds

In [53]:
# Find medications in our data that belong to a certain category, as defined in this functions:
opioids_meds, benzos_meds, antipsychotics_meds = opioid_benzo_antipsychotic_list()
# Initialize a list of general equivalency, i.e. for each type of medication
opioids_ratios, benzos_ratios, antipsychotics_ratios = general_medication_dose_equivalencies()
# Create a dictionary of equivalency for doses for specific medications (MedicationDSC) contained in our data:
opioids_meds_ratios, benzos_meds_ratios, antipsychotics_meds_ratios = specific_medication_dose_equivalencies()
# And finally, compute the actual equivalent doses for all the entries:
meds = compute_equivalencies(meds)

In [54]:
meds.head()

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,IsInfusion,Unit,Dose,MedicationDSCNew,MedicationGenericName,Dose_Opioid_Equivalent,Dose_Benzo_Equivalent,Dose_Antipsychotics_Equivalent
38153,664939918.0,3880359,3.301282e+09,11150.0,propofol (diprivan) infusion 10 mg/ml,2020-03-26 20:43:00.0000000,2020-03-26 20:43:00,2020-03-26 22:01:00,2020-03-26 22:01:00,6.0,New Bag,NaN,NaN,NaN,NaN,NaN,MAR,70.0,NaN,NaN,NaN,False,mg/hr,498.96,propofol_(diprivan)_infusion_10_mgperml,propofol,NaN,79833.6,NaN
38219,664939923.0,3880359,3.301282e+09,172732.0,midazolam (pf) 1 mg/ml injection solution,2020-03-26 21:44:00.0000000,2020-03-26 22:30:00,2020-03-26 22:01:00,2020-03-26 22:01:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,MAR,2.5,NaN,NaN,Once,False,mg,2.50,midazolam_(pf)_1_mgperml_injection_solution,midazolam,NaN,10.0,NaN
38266,664939925.0,3880359,3.301282e+09,95811.0,rocuronium 10 mg/ml intravenous solution,2020-03-26 21:47:00.0000000,2020-03-26 22:45:00,2020-03-26 22:02:00,2020-03-26 22:02:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,MAR,50.0,NaN,NaN,Once,False,mg,50.00,rocuronium_10_mgperml_intravenous_solution,rocuronium,NaN,NaN,NaN
38160,664939927.0,3880359,3.301282e+09,95811.0,rocuronium 10 mg/ml intravenous solution,2020-03-26 22:20:00.0000000,2020-03-26 23:15:00,2020-03-26 22:47:00,2020-03-26 22:47:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,MAR,50.0,NaN,NaN,Once,False,mg,50.00,rocuronium_10_mgperml_intravenous_solution,rocuronium,NaN,NaN,NaN
39041,664939926.0,3880359,3.301282e+09,11150.0,propofol (diprivan) infusion 10 mg/ml,2020-03-26 22:16:00.0000000,2020-03-26 22:16:00,2020-03-26 22:47:00,2020-03-26 22:47:00,6.0,New Bag,NaN,NaN,NaN,NaN,NaN,MAR,70.0,NaN,NaN,NaN,False,mg/hr,498.96,propofol_(diprivan)_infusion_10_mgperml,propofol,NaN,79833.6,NaN


In [55]:
meds[pd.notna(meds['Dose_Opioid_Equivalent'])].MedicationGenericName.unique()

array(['hydromorphone', 'oxycodone', 'fentanyl', 'morphine', 'methadone',
       'tramadol', 'meperidine'], dtype=object)

In [56]:
meds[pd.notna(meds['Dose_Benzo_Equivalent'])].MedicationGenericName.unique()

array(['propofol', 'midazolam', 'lorazepam', 'dexmedetomidine',
       'diazepam', 'clonazepam'], dtype=object)

In [57]:
meds[pd.notna(meds['Dose_Antipsychotics_Equivalent'])].MedicationGenericName.unique()

array(['quetiapine', 'haloperidol', 'olanzapine'], dtype=object)

In [58]:
meds[np.all(pd.isna([meds['Dose_Opioid_Equivalent'],
                     meds['Dose_Benzo_Equivalent'], 
                     meds['Dose_Antipsychotics_Equivalent']]), axis=0)].MedicationGenericName.unique()

array(['rocuronium', 'ketamine', 'melatonin', 'cisatracurium',
       'vecuronium', 'phenobarbital', 'gabapentin', 'clonidine',
       'isoflurane', 'methylphenidate', 'caffeine', 'chlorpromazine',
       'valproic acid', 'modafinil', 'aripiprazole', 'amantadine'],
      dtype=object)

In [59]:
def compute_drug_class_sum(meds_ts_df_mrn):

    for med in meds_ts_df_mrn.columns:
        if '_sum' in med: continue
        if 'placebo' in med: continue
        generic_name = [x for x in pd.unique(meds_to_study) if x in med]
        if (len(generic_name) == 0) & ('_' in med):
            generic_name = [x for x in meds_to_study if x in med.replace('_', ' ')]
        if len(generic_name) == 0:
            print(f'No generic medication categories found for {med}: {generic_name}. Skip.')
            continue
        if len(generic_name)>1:    
            # first, remove medication name that are a substring of another name:
            for x in generic_name:
                if not x in generic_name: 
                    continue #this is werid but has to be done because .remove() is inplace operation.
                if any([x in y for y in set(generic_name) - set([x])]):
                    generic_name.remove(x)
        if len(generic_name)>1:            
            print(f'Multiple generic medication categories found for {med}: {generic_name}. Take the first.')
        generic_name = generic_name[0]
        if not generic_name +'_sum' in meds_ts_df_mrn.columns:
            meds_ts_df_mrn[generic_name + '_sum'] = 0
        meds_ts_df_mrn[generic_name + '_sum'] += meds_ts_df_mrn[med]
    
    return meds_ts_df_mrn

In [61]:
def timeseries_for_order(meds_ts_df_mrn, df_order, medname_original_new):
    
    order_id = df_order.OrderID.iloc[0]
    med = df_order[medname_col].unique()
    unit = df_order.Unit.dropna().unique()
    route = df_order.RouteDSC.dropna().unique()
    assert len(med) <= 1, f'Different Medication Names within the same OrderID {order_id}'
    assert len(route) <= 1, f'Different Routes within the same OrderID {order_id}'
    assert len(unit) <= 1, f'Different Units within the same OrderID {order_id}'
    
    if len(med) > 0: med = med[0]
    if len(unit) > 0: unit = unit[0]
    route = route[0] if len(route) > 0 else 'undefined'
    med_name = medname_original_new[med]

    type_order = None
    if (unit == 'mg/hr') & ~('patch' in med): # (route in ['Intravenous', 'Epidural', 'Intravenous (PCA General)']):
        type_order = 'infusion_iv_continuous'
    elif (unit == 'mg/hr') & ('patch' in med):
        type_order = 'patch'
    elif unit == 'mg': # either bolus or tablet
        type_order = 'discrete'
    elif med_name == 'DEXMEDETOMIDINE_OR_PLACEBO_(2017P000090)'.lower():
        type_order = 'icu_sleep_studydrug'
    else:
        print(f'Undefined Type of Med: {df_order.iloc[0]}')


    if type_order == 'infusion_iv_continuous':
        
        tmp = pd.DataFrame(index = meds_ts_df_mrn.index, columns = ['tmp'])
        tmp.iloc[0] = 0
        # infusions are saved in mg/min, so that it matches the timeseries with 1 min sample rate.
        if len(df_order[df_order['Dose']>0]) == 0:
                return meds_ts_df_mrn
        start_time_order = df_order[df_order['Dose']>0].iloc[0].MedicationTakenDTS
        end_time_order = df_order.iloc[-1].OrderEndDTS
        # make sure there's no other entry yet in this timerange for this med:
#         if np.sum(meds_ts_df_mrn.loc[start_time_order : end_time_order, med_name].dropna()) > 0:
#             print(f'Overlapping Order Start and End for: MRN={mrn}, Medication Name={med}, start_time_order={start_time_order}, end_time_order={end_time_order}')
        #  Correct end time order if last entry is set to 0:
        if df_order.iloc[-1].Dose == 0:
            end_time_order = df_order.iloc[-1].MedicationTakenDTS
        df_order = df_order[(df_order.MedicationTakenDTS >= start_time_order) & (df_order.MedicationTakenDTS <= end_time_order)]
        if len(df_order[df_order['Dose']>0]) == 0:
                return meds_ts_df_mrn
        for jloc, row in df_order.iterrows():
            tmp.loc[row.MedicationTakenDTS, 'tmp'] = row.Dose / 60 # mg/min
        # time series with forward-propagate/fill NA values for the continuous meds:
        tmp.loc[end_time_order, 'tmp'] = 0
        tmp['tmp'].fillna(method='ffill', inplace=True)   
        # add it to 'general' med_name:
        meds_ts_df_mrn[med_name] += tmp['tmp'].values
        if medname_col == 'MedicationGenericName':
            # if we are working with generic drug names here, save it as oral or parenteral as well:
            route_category = get_route_from_medname(row.MedicationDSC)
            meds_ts_df_mrn[med_name + '_' + route_category] += tmp['tmp'].values
    
    elif type_order == 'patch':

        if not 'Patch Applied' in df_order.MARActionDSC.values:
            print('no patch applied?')

        tmp = pd.DataFrame(index = meds_ts_df_mrn.index, columns = ['tmp'])
        tmp.iloc[0] = 0

        start_time_order = df_order[df_order['MARActionDSC'] == 'Patch Applied'].iloc[0].MedicationTakenDTS
        end_time_order = df_order.iloc[-1].OrderEndDTS
        if df_order.iloc[-1].MARActionDSC == 'Patch Removed':
            end_time_order = df_order.iloc[-1].MedicationTakenDTS

#         meds_ts_df_mrn[med_name].loc[start_time_order : end_time_order] = np.nan

        for jloc, row in df_order.iterrows():
            if row.MARActionDSC in ['Patch Removed', 'Due']:
                assert row.Dose == 0
            tmp.loc[row.MedicationTakenDTS, 'tmp'] = row.Dose / 60

        # if last entry has dose>0, there is no end indicated. use maximum duration derived from DiscreteFrequency. 
        # I think this is necessary because some Orders for patches are for a month, without any entry after a few days any more.
        if row.Dose > 0:
            try:
                duration = np.float(re.search('\d+', row.DiscreteFrequencyDSC)[0])
                unit_duration = [x for x in ['minute', 'hour', 'day'] if x in row.DiscreteFrequencyDSC.lower()][0]
            except:
                print(f'Patch {row.DiscreteFrequencyDSC} max duration parsing failed.')
                return meds_ts_df_mrn
                
            if unit_duration == 'minute':
                duration = duration/60
            elif unit_duration == 'hour':
                duration = duration
            elif unit_duration == 'day':
                duration = duration*24
            else:
                print(f'Unexpected duration unit for Patch {unit_duration}')

            end_dt = row.MedicationTakenDTS + timedelta(hours=duration)
            end_dt = np.min([meds_ts_df_mrn.index[-1], end_dt])
            tmp.loc[end_dt, 'tmp'] = 0
        
        tmp['tmp'].fillna(method='ffill', inplace=True)   
        # add it to 'general' med_name:
        meds_ts_df_mrn[med_name] += tmp['tmp'].values
        if medname_col == 'MedicationGenericName':
            # if we are working with generic drug names here, save it as oral or parenteral as well:
            route_category = get_route_from_medname(row.MedicationDSC)
            meds_ts_df_mrn[med_name + '_' + route_category] += tmp['tmp'].values

            

    elif type_order == 'discrete':

        for jloc, row in df_order.iterrows():   
            meds_ts_df_mrn.loc[row.MedicationTakenDTS, med_name] += row.Dose
            
            if medname_col == 'MedicationGenericName':
                # if we are working with generic drug names here, save it as oral or parenteral as well:
                route_category = get_route_from_medname(row.MedicationDSC)
                meds_ts_df_mrn.loc[row.MedicationTakenDTS, med_name + '_' + route_category] += row.Dose

    elif type_order == 'icu_sleep_studydrug':
        
        tmp = pd.DataFrame(index = meds_ts_df_mrn.index, columns = ['tmp'])
        tmp.iloc[0] = 0

        for jloc, row in df_order.iterrows():
            if row.Dose > 0:
                if row.DurationUnitDSC == 'Minutes':
                    duration_minutes = np.float(row.DurationNBR)
                elif row.DurationUnitDSC == 'Hours':
                    duration_minutes = np.float(row.DurationNBR) * 60
                else:
                    print(f'Unexpected DurationUnitDSC for ICU-Sleep study drug: {row.DurationUnitDSC}')

                end_dt = row.MedicationTakenDTS + timedelta(minutes=duration_minutes)
#                 meds_ts_df_mrn.loc[row.MedicationTakenDTS, med_name] = row.Dose / 60 # ml/min
                tmp.loc[row.MedicationTakenDTS, 'tmp'] = row.Dose / 60

                if pd.isna(meds_ts_df_mrn.loc[end_dt, med_name]):
#                     meds_ts_df_mrn.loc[end_dt, med_name] = 0 # set definitive end point.
                    tmp.loc[end_dt, 'tmp'] = 0 # set definitive end point.

            else:
                tmp.loc[row.MedicationTakenDTS, 'tmp'] = 0
        tmp['tmp'].fillna(method='ffill', inplace=True)   
        # add it to 'general' med_name:
        meds_ts_df_mrn[med_name] += tmp['tmp'].values
        if medname_col == 'MedicationGenericName':
            # if we are working with generic drug names here, save it as oral or parenteral as well:
            route_category = get_route_from_medname(row.MedicationDSC)
            meds_ts_df_mrn[med_name + '_' + route_category] += tmp['tmp'].values
            
    return meds_ts_df_mrn


def compute_equivalencies_timeseries(meds_timeseries):

    meds_timeseries['opioids_sum'] = 0
    meds_timeseries['benzos_sum'] = 0
    meds_timeseries['antipsychotics_sum'] = 0
    equivalency_ratios_list = [opioids_meds_ratios, benzos_meds_ratios, antipsychotics_meds_ratios]
    equivalency_names = ['opioids_sum', 'benzos_sum', 'antipsychotics_sum']

    for equi_classname, equi_ratio in list(zip(equivalency_names, equivalency_ratios_list)):
        for equi_med in equi_ratio:
            if equi_med in meds_timeseries.columns:
                meds_timeseries[equi_classname] = meds_timeseries[equi_classname] + meds_timeseries[equi_med] / equi_ratio[equi_med]
                        
    return meds_timeseries

def compute_equivalencies_timeseries_generic(meds_timeseries):

    meds_timeseries['opioids_sum'] = 0
    meds_timeseries['benzos_sum'] = 0
    meds_timeseries['antipsychotics_sum'] = 0
    equivalency_ratios_list = [opioids_ratios, benzos_ratios, antipsychotics_ratios]
    equivalency_names = ['opioids_sum', 'benzos_sum', 'antipsychotics_sum']

    for equi_classname, equi_ratio in list(zip(equivalency_names, equivalency_ratios_list)):
        for equi_med in equi_ratio:
            if equi_med in meds_timeseries.columns.str.lower():
                colname_tmp = [x for x in meds_timeseries.columns if equi_med in x.lower()][0]
                if sum(meds_timeseries[colname_tmp]) == 0: continue
                meds_timeseries[equi_classname] = meds_timeseries[equi_classname] + meds_timeseries[colname_tmp] / equi_ratio[equi_med]
                        
    return meds_timeseries

def create_timeseries_df_for_subject(df_mrn):
    ''' 
    Input: Medication data for one subject, after running pipeline that computes 'Dose' and 'Unit' columns.
    Ouput: dataframe in 1-minute resolution: Discrete medications in mg, Continuous in mg/min.
    '''

    min_date = np.nanmin([df_mrn.OrderStartDTS.values, df_mrn.MedicationTakenDTS.values])
    min_date = pd.Timestamp(min_date) - timedelta(minutes=1)
    max_date = np.nanmax([df_mrn.OrderEndDTS.values, df_mrn.MedicationTakenDTS.values])

    # initialize new timeseries-based df for this MRN:
    meds_ts_df_mrn = pd.DataFrame(index=pd.date_range(min_date, max_date, freq = fs_med)) 

    medname_original_new = init_medname_dict(df_mrn)

    for med_name_orig in medname_original_new:
        meds_ts_df_mrn[medname_original_new[med_name_orig]] = 0
        meds_ts_df_mrn[medname_original_new[med_name_orig] + '_oral'] = 0
        meds_ts_df_mrn[medname_original_new[med_name_orig] + '_parenteral'] = 0
        meds_ts_df_mrn[medname_original_new[med_name_orig] + '_transdermal'] = 0

    order_ids = pd.unique(df_mrn.OrderID)
    
    for order_id in order_ids:
        df_order = df_mrn[df_mrn.OrderID == order_id].copy()
        meds_ts_df_mrn = timeseries_for_order(meds_ts_df_mrn, df_order, medname_original_new)
    
#     # time series with forward-propagate/fill NA values for the continuous meds:
#     for col in meds_ts_df_mrn.columns:
        
    if medname_col == 'MedicationGenericName':
        # equivalencies for Generics:
        meds_ts_df_mrn = compute_equivalencies_timeseries_generic(meds_ts_df_mrn)

    else:
        # print('Generic Med Type Summation')
        meds_ts_df_mrn =  compute_drug_class_sum(meds_ts_df_mrn)
        # equivalencies for specific meds:
        meds_ts_df_mrn = compute_equivalencies_timeseries(meds_ts_df_mrn)


    return meds_ts_df_mrn
    

def create_hdr():
    
    hdr = {}
    if 'study_id' in row_subject:
        hdr['study_id'] = int(row_subject.study_id)
    if 'mrn' in row_subject:
        hdr['MRN'] = int(row_subject.mrn)
    if 'PatientID' in row_subject:
        hdr['PatientID'] = row_subject.PatientID
    hdr['fs'] = fs_med
    hdr['start_datetime'] = np.array([meds_ts_df_mrn.index[0].year,
                                     meds_ts_df_mrn.index[0].month,
                                     meds_ts_df_mrn.index[0].day,
                                     meds_ts_df_mrn.index[0].hour,
                                     meds_ts_df_mrn.index[0].minute,
                                     meds_ts_df_mrn.index[0].second,
                                     meds_ts_df_mrn.index[0].microsecond])
    
    return hdr

In [62]:
edw_data.head()

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,IsInfusion,Unit,Dose,MedicationDSCNew
38153,664939918.0,3880359,3.301282e+09,11150.0,propofol (diprivan) infusion 10 mg/ml,2020-03-26 20:43:00.0000000,2020-03-26 20:43:00,2020-03-26 22:01:00,2020-03-26 22:01:00,6.0,New Bag,NaN,NaN,NaN,NaN,NaN,MAR,70.0,NaN,NaN,NaN,False,mcg/kg/min,70.0,propofol_(diprivan)_infusion_10_mgperml
38219,664939923.0,3880359,3.301282e+09,172732.0,midazolam (pf) 1 mg/ml injection solution,2020-03-26 21:44:00.0000000,2020-03-26 22:30:00,2020-03-26 22:01:00,2020-03-26 22:01:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,MAR,2.5,NaN,NaN,Once,False,mg,2.5,midazolam_(pf)_1_mgperml_injection_solution
38266,664939925.0,3880359,3.301282e+09,95811.0,rocuronium 10 mg/ml intravenous solution,2020-03-26 21:47:00.0000000,2020-03-26 22:45:00,2020-03-26 22:02:00,2020-03-26 22:02:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,MAR,50.0,NaN,NaN,Once,False,mg,50.0,rocuronium_10_mgperml_intravenous_solution
38160,664939927.0,3880359,3.301282e+09,95811.0,rocuronium 10 mg/ml intravenous solution,2020-03-26 22:20:00.0000000,2020-03-26 23:15:00,2020-03-26 22:47:00,2020-03-26 22:47:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,MAR,50.0,NaN,NaN,Once,False,mg,50.0,rocuronium_10_mgperml_intravenous_solution
39041,664939926.0,3880359,3.301282e+09,11150.0,propofol (diprivan) infusion 10 mg/ml,2020-03-26 22:16:00.0000000,2020-03-26 22:16:00,2020-03-26 22:47:00,2020-03-26 22:47:00,6.0,New Bag,NaN,NaN,NaN,NaN,NaN,MAR,70.0,NaN,NaN,NaN,False,mcg/kg/min,70.0,propofol_(diprivan)_infusion_10_mgperml


In [63]:
def unit_correction_for_multiple_units_within_one_order(meds):

    for order_id in meds.OrderID.unique():
        df_order = meds.loc[meds.OrderID == order_id]
        unit = df_order.Unit.dropna().unique()
        if len(unit) > 1:
            print(f'Unit Correction due to multiple units in OrderID {order_id}')
            assert(all(np.isin(['mg', 'mg/hr'], unit)))
            if all(pd.notna(df_order.loc[df_order.Unit=='mg'].DurationNBR)):
                for jloc, row in df_order.iterrows():
                    if row.Unit == 'mg':
                        meds.loc[jloc, 'Dose'] = row.Dose/row.DurationNBR
                        meds.loc[jloc, 'Unit'] = 'mg/hr'
                        
    return meds

meds = unit_correction_for_multiple_units_within_one_order(meds)

In [64]:
meds_all_output_dir

'/media/ssd2/Covid19_Delirium/medications_all_timeseries'

In [65]:
if project == 'covid_sedation':
    meds_all_output_dir_hourly_h5 = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries_hourly_h5'
    meds_all_output_dir_hourly_csv = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries_hourly_csv'
    meds_all_output_dir_hourly_wroute_csv = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries_hourly_w_route_csv'
    medications_hourly_figures_png = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_hourly_figures_png'
    medications_hourly_figures_pdf = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_hourly_figures_pdf'

elif project == 'covid_delirium':
    meds_all_output_dir_hourly_h5 = '/media/mad3/Projects/Wolfgang/Covid19_Delirium/medications_all_timeseries_hourly_h5'
    meds_all_output_dir_hourly_csv = '/media/mad3/Projects/Wolfgang/Covid19_Delirium/medications_all_timeseries_hourly_csv'
    meds_all_output_dir_hourly_wroute_csv = '/media/mad3/Projects/Wolfgang/Covid19_Delirium/medications_all_timeseries_hourly_w_route_csv'
    medications_hourly_figures_png = '/media/mad3/Projects/Wolfgang/Covid19_Delirium/medications_hourly_figures_png'
    medications_hourly_figures_pdf = '/media/mad3/Projects/Wolfgang/Covid19_Delirium/medications_hourly_figures_pdf'

for directory in [meds_all_output_dir_hourly_h5, meds_all_output_dir_hourly_csv, meds_all_output_dir_hourly_wroute_csv, 
                 medications_hourly_figures_png, medications_hourly_figures_pdf]:
    if not os.path.exists(directory):
        os.makedirs(directory)

In [66]:
meds.loc[meds.MedicationDSC == 'zz ims template', 'MedicationDSC'] = meds.loc[meds.MedicationDSC == 'zz ims template', 'MedicationGenericName'].str.lower() + ' ' + meds.loc[meds.MedicationDSC == 'zz ims template', 'RouteDSC'].str.lower()

In [67]:
meds.loc[meds.MedicationDSC == 'Clonidine Oral', 'MedicationDSC'] = 'clonidine oral'

In [68]:
def meds_plot_2(meds_ts_df_mrn_hourly, patient_id = ''):

    fig, ax = plt.subplots(len(meds_ts_df_mrn_hourly.columns), 1, sharex=True, figsize=(10, max(3, len(meds_ts_df_mrn_hourly.columns)*1.5)));

    for i, med in enumerate(meds_ts_df_mrn_hourly.columns):
        
        if med == 'GCS':
            color = 'green'
        
        elif (med.lower() in opioids_meds) | (med=='opioids_sum'):
            color = 'red'
        elif (med.lower() in benzos_meds) | (med=='benzos_sum'):
            color = 'orange'
        elif (med.lower() in antipsychotics_meds) | (med=='antipsychotics_sum'):
            color = 'red'
        else:
            color = 'black'
        
        ax[i].scatter(meds_ts_df_mrn_hourly.index[pd.notna(meds_ts_df_mrn_hourly[med])], meds_ts_df_mrn_hourly[med].dropna().astype(float), color=color, s=5);
        ax[i].plot(meds_ts_df_mrn_hourly.index[pd.notna(meds_ts_df_mrn_hourly[med])], meds_ts_df_mrn_hourly[med].dropna().astype(float), color=color, alpha = 0.4);
        ax[i].set_ylabel(med);
        
    ax[0].set_title('PatientID='+str(patient_id) + ', MRN=' + str(int(mrn)) +', '+ title + ' hourly doses meds (mg)');
    
    return fig

In [69]:
meds.head(1)

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,IsInfusion,Unit,Dose,MedicationDSCNew,MedicationGenericName,Dose_Opioid_Equivalent,Dose_Benzo_Equivalent,Dose_Antipsychotics_Equivalent
38153,664939918.0,3880359,3.301282e+09,11150.0,propofol (diprivan) infusion 10 mg/ml,2020-03-26 20:43:00.0000000,2020-03-26 20:43:00,2020-03-26 22:01:00,2020-03-26 22:01:00,6.0,New Bag,NaN,NaN,NaN,NaN,NaN,MAR,70.0,NaN,NaN,NaN,False,mg/hr,498.96,propofol_(diprivan)_infusion_10_mgperml,propofol,NaN,79833.6,NaN


In [70]:
if 0:### check what's happening for Z6844218 
    meds.loc[(meds.MRN == 4406399) & (meds.MedicationTakenDTS < '2020-04-16') & (meds.MedicationID == 700001)]

In [71]:
# %matplotlib agg
%matplotlib inline

In [72]:
def get_aggregation_dict(meds_ts_df_mrn):
    
    aggregation_dict = {}
    for col in meds_ts_df_mrn.columns:
        if not 'GCS' in col:
            aggregation_dict[col] = 'sum'
        elif 'GCS' in col:
            aggregation_dict[col] = 'min'
        else:
            print('bug.')
            
    return aggregation_dict


In [73]:
meds_c = meds.copy()

In [74]:
meds = meds_c.copy()

In [75]:
save = True
overwrite = True
fs_med =  '1min'

meds_ts_df_mrn_all = []

for jloc_subject, row_subject in tqdm(cohort.iterrows()):

    mrn = row_subject.mrn
    meds_mrn = meds[meds.MRN == mrn]
    patient_id = mrn
    
    if meds_mrn.shape[0] == 0:
        continue
        
    meds_ts_df_mrn = create_timeseries_df_for_subject(meds_mrn)

    if project == 'icu-sleep':
        hdr = create_hdr() 
        file_output_path = os.path.join(meds_all_output_dir, 'icu_' + str(row_subject.study_id) + '_meds.h5')
        
    elif project == 'sample':
        row_subject['study_id'] = 0
        hdr = create_hdr() 
        file_output_path = os.path.join(meds_all_output_dir, str(row_subject.mrn) + '_meds.h5')

    elif project == 'covid_sedation':
        
        patient_id = row_subject.PatientID
        hdr = create_hdr() 
        file_output_path = os.path.join(meds_all_output_dir, str(patient_id) + '_meds.h5')
        file_output_path_hourly = os.path.join(meds_all_output_dir_hourly_h5, str(patient_id) + '_meds.h5')
        file_output_path_hourly_csv = os.path.join(meds_all_output_dir_hourly_csv, str(patient_id) + '_meds.csv')
        file_output_path_hourly_wroute_csv = os.path.join(meds_all_output_dir_hourly_wroute_csv, str(patient_id) + '_meds.csv')

    elif project == 'covid_delirium':
        
        hdr = create_hdr() 
        file_output_path = os.path.join(meds_all_output_dir, str(mrn) + '_meds.h5')
        file_output_path_hourly = os.path.join(meds_all_output_dir_hourly_h5, str(mrn) + '_meds.h5')
        file_output_path_hourly_csv = os.path.join(meds_all_output_dir_hourly_csv, str(mrn) + '_meds.csv')
        file_output_path_hourly_wroute_csv = os.path.join(meds_all_output_dir_hourly_wroute_csv, str(mrn) + '_meds.csv')


    if 0:
        
        gcs_data_mrn = gcs_data[gcs_data.MRN == mrn]
        meds_ts_df_mrn = meds_ts_df_mrn.join(gcs_data_mrn['GCS'].astype(float), how='left')
        
        gcs_data_motor_mrn = gcs_data_motor[gcs_data_motor.MRN == mrn]
        meds_ts_df_mrn = meds_ts_df_mrn.join(gcs_data_motor_mrn['GCS_Motor'].astype(float), how='left')
        
        gcs_data_eye_mrn = gcs_data_eye[gcs_data_eye.MRN == mrn]
        meds_ts_df_mrn = meds_ts_df_mrn.join(gcs_data_eye_mrn['GCS_Eye'].astype(float), how='left')
        
        gcs_data_verbal_mrn = gcs_data_verbal[gcs_data_verbal.MRN == mrn]
        meds_ts_df_mrn = meds_ts_df_mrn.join(gcs_data_verbal_mrn['GCS_Verbal'].astype(float), how='left')
    
    if save:
        write_to_hdf5_file(meds_ts_df_mrn, file_output_path, hdr=hdr, default_dtype='float32', overwrite=overwrite)
    meds_ts_df_mrn_all.append(meds_ts_df_mrn)
    if 1:
        # hourly summary:
        aggregation_dict = get_aggregation_dict(meds_ts_df_mrn)
                
        meds_ts_df_mrn_hourly = meds_ts_df_mrn.resample('1h', label='right').agg(aggregation_dict)
        hdr['fs'] = '1h'
#         gcs_data:
#         gcs_data_mrn = gcs_data[gcs_data.MRN == mrn]
#         gcs_data_mrn.rename({'MeasureTXT': 'GCS'}, axis=1, inplace=True)
        
        
#         for gcs_type in ['GCS', 'GCS_Motor', 'GCS_Eye', 'GCS_Verbal']:
#             gcs_data_mrn_hourly = gcs_data_mrn[[gcs_type]].astype(float).resample('1h').min()
#             if gcs_type in meds_ts_df_mrn_hourly.columns:
#                 meds_ts_df_mrn_hourly.drop(gcs_type, axis=1, inplace=True)
#             meds_ts_df_mrn_hourly = meds_ts_df_mrn_hourly.join(gcs_data_mrn_hourly[gcs_type], how='left')
        meds_ts_df_mrn_hourly = meds_ts_df_mrn_hourly.round(10)
#         meds_ts_df_mrn_hourly.
#         meds_ts_df_mrn_hourly inplace=True)
        if save:
            write_to_hdf5_file(meds_ts_df_mrn_hourly, file_output_path_hourly, hdr=hdr, default_dtype='float32', overwrite=overwrite)
            meds_ts_df_mrn_hourly.reset_index().rename({'index': 'datetime'}, axis=1).to_csv(file_output_path_hourly_wroute_csv, index=False)

    if project == 'covid_delirium':
        meds_mrn_totalsum = meds_ts_df_mrn_hourly.sum()
        for med_tmp in cohort.columns[5:]:
            if med_tmp + '_sum' in meds_mrn_totalsum.keys():
                cohort.loc[jloc_subject, med_tmp] = meds_mrn_totalsum[med_tmp + '_sum']
            elif med_tmp in meds_mrn_totalsum.keys():
                cohort.loc[jloc_subject, med_tmp] = meds_mrn_totalsum[med_tmp]
            else:
                cohort.loc[jloc_subject, med_tmp] = 0

    if 0:
        
        cols = []
        for col_tmp in meds_ts_df_mrn.columns:
            if not any([x in col_tmp for x in ['_oral', '_parenteral', 'transdermal']]):
                if sum(meds_ts_df_mrn[col_tmp].dropna().astype(float)) > 0:
                    cols.append(col_tmp)
        cols_sorted = [x for x in cols if x.lower() in opioids_meds] + [x for x in cols if x.lower() in benzos_meds] + [x for x in cols if x.lower() in antipsychotics_meds]
        cols_sorted = cols_sorted + list(set(cols) - set(['GCS', 'GCS_Motor', 'GCS_Eye', 'GCS_Verbal', 'opioids_sum', 'benzos_sum', 'antipsychotics_sum']) - set(cols_sorted))
        cols_sorted += ['opioids_sum', 'benzos_sum', 'antipsychotics_sum', 'GCS', 'GCS_Motor', 'GCS_Eye', 'GCS_Verbal']
        cols_sorted = [x for x in cols_sorted if x in cols]
        if save:
            meds_ts_df_mrn_hourly[cols_sorted].reset_index().rename({'index': 'datetime'}, axis=1).to_csv(file_output_path_hourly_csv, index=False)

        title = ''
        fig = meds_plot_2(meds_ts_df_mrn_hourly[cols_sorted], patient_id = patient_id)
        if save:
            fig.savefig(os.path.join(medications_hourly_figures_png, str(patient_id) + '_meds.png'), dpi=500);
            fig.savefig(os.path.join(medications_hourly_figures_pdf, str(patient_id) + '_meds.pdf'), dpi=500);
        plt.close('all');

17it [00:33,  1.66s/it]

Patch Weekly max duration parsing failed.


56it [01:46,  1.68s/it]

Patch Weekly max duration parsing failed.


67it [02:16,  2.04s/it]


In [79]:
if project == 'covid_delirium':
    cohort.to_csv('covid_delirium_meds.csv', index=False)

In [76]:
[x for x in meds.MedicationDSC.unique() if 'isoflurane' in x.lower()]

['isoflurane 99.9 % inhalation liquid']

In [77]:
meds.loc[meds.MedicationDSC == 'olanzapine 10 mg tablet']

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,IsInfusion,Unit,Dose,MedicationDSCNew,MedicationGenericName,Dose_Opioid_Equivalent,Dose_Benzo_Equivalent,Dose_Antipsychotics_Equivalent
58957,677123619.0,6226515,3.306511e+09,17937.0,olanzapine 10 mg tablet,2020-05-17 16:35:00.0000000,2020-05-17 21:00:00,2020-05-18 08:45:00,2020-05-17 20:02:00,1.0,Given,Nasogastric Tube,35.0,NaN,NaN,NaN,MAR,10.0,NaN,NaN,Nightly,False,mg,10.0,olanzapine_10_mg_tablet,olanzapine,NaN,NaN,100.0


In [81]:
cohort.sum();

In [67]:
print(0.025/60)
print(0.05/60)
meds_ts_df_mrn[['Fentanyl']].iloc[:200];

0.0004166666666666667
0.0008333333333333334


In [71]:
meds_ts_df_mrn_hourly

,Midazolam,Midazolam_oral,Midazolam_parenteral,Midazolam_transdermal,Propofol,Propofol_oral,Propofol_parenteral,Propofol_transdermal,Fentanyl,Fentanyl_oral,Fentanyl_parenteral,Fentanyl_transdermal,Hydromorphone,Hydromorphone_oral,Hydromorphone_parenteral,Hydromorphone_transdermal,Ketamine,Ketamine_oral,Ketamine_parenteral,Ketamine_transdermal,opioids_sum,benzos_sum,antipsychotics_sum,GCS,GCS_Motor,GCS_Eye,GCS_Verbal
2020-04-15 13:00:00,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.000000,0.000000,0,NaN,NaN,NaN,NaN
2020-04-15 14:00:00,4.0,0,4.0,0,108.083333,0,108.083333,0,0.029583,0,0.029583,0,0.0,0,0.0,0,0.0,0,0.0,0,2.958333,1.675521,0,3.0,1.0,1.0,1.0
2020-04-15 15:00:00,0.0,0,0.0,0,166.950000,0,166.950000,0,0.050000,0,0.050000,0,0.0,0,0.0,0,0.0,0,0.0,0,5.000000,1.043438,0,NaN,NaN,NaN,NaN
2020-04-15 16:00:00,0.0,0,0.0,0,293.000000,0,293.000000,0,0.071667,0,0.071667,0,1.0,0,1.0,0,0.0,0,0.0,0,13.833333,1.831250,0,NaN,NaN,NaN,NaN
2020-04-15 17:00:00,0.0,0,0.0,0,293.000000,0,293.000000,0,0.075000,0,0.075000,0,1.0,0,1.0,0,0.0,0,0.0,0,14.166667,1.831250,0,3.0,NaN,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-04-28 05:00:00,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.000000,0.000000,0,3.0,NaN,1.0,1.0
2020-04-28 06:00:00,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.000000,0.000000,0,NaN,NaN,NaN,NaN
2020-04-28 07:00:00,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.000000,0.000000,0,NaN,NaN,NaN,NaN
2020-04-28 08:00:00,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.0,0,0.0,0,0.0,0,0.0,0,0.000000,0.000000,0,NaN,NaN,NaN,NaN


In [60]:
meds_ts_df_mrn_hourly[['Fentanyl', 'Fentanyl_oral', 'Fentanyl_parenteral', 'Fentanyl_transdermal', 'opioids_sum']]

,Fentanyl,Fentanyl_oral,Fentanyl_parenteral,Fentanyl_transdermal,opioids_sum
2020-04-15 12:00:00,0.000000,0,0.000000,0,0.000000
2020-04-15 13:00:00,0.029583,0,0.029583,0,2.958333
2020-04-15 14:00:00,0.050000,0,0.050000,0,5.000000
2020-04-15 15:00:00,0.071667,0,0.071667,0,13.833333
2020-04-15 16:00:00,0.075000,0,0.075000,0,14.166667
...,...,...,...,...,...
2020-04-28 04:00:00,0.000000,0,0.000000,0,0.000000
2020-04-28 05:00:00,0.000000,0,0.000000,0,0.000000
2020-04-28 06:00:00,0.000000,0,0.000000,0,0.000000
2020-04-28 07:00:00,0.000000,0,0.000000,0,0.000000


In [355]:
meds_ts_df_mrn, hdr = load_sleep_data('/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries/3337103.0_meds.h5', idx_to_datetime=1)

In [448]:
meds_ts_df_mrn

,Midazolam,Midazolam_oral,Midazolam_parenteral,Midazolam_transdermal,Propofol,Propofol_oral,Propofol_parenteral,Propofol_transdermal,Fentanyl,Fentanyl_oral,Fentanyl_parenteral,Fentanyl_transdermal,Hydromorphone,Hydromorphone_oral,Hydromorphone_parenteral,Hydromorphone_transdermal,Ketamine,Ketamine_oral,Ketamine_parenteral,Ketamine_transdermal,Dexmedetomidine,Dexmedetomidine_oral,Dexmedetomidine_parenteral,Dexmedetomidine_transdermal,Lorazepam,Lorazepam_oral,Lorazepam_parenteral,Lorazepam_transdermal,Methadone,Methadone_oral,Methadone_parenteral,Methadone_transdermal,Diazepam,Diazepam_oral,Diazepam_parenteral,Diazepam_transdermal,Gabapentin,Gabapentin_oral,Gabapentin_parenteral,Gabapentin_transdermal,Clonidine,Clonidine_oral,Clonidine_parenteral,Clonidine_transdermal,opioids_sum,benzos_sum,antipsychotics_sum
2020-04-07 14:59:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
2020-04-07 15:00:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
2020-04-07 15:01:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
2020-04-07 15:02:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
2020-04-07 15:03:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-18 14:32:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
2020-06-18 14:33:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
2020-06-18 14:34:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0
2020-06-18 14:35:00,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0


In [1]:
benzo

NameError: name 'benzo' is not defined

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:3: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  This is separate from the ipykernel package so we can avoid doing imports until


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [527]:
col_tmp

'GCS'

In [509]:
# med = 'hydroxychloroquine_200_mg_tablet'
med = 'propofol_10_mgperml_intravenous_emulsion'
med = 'Propofol'
med = 'benzos_sum'
# med = 'opioids_sum'
med = 'antipsychotics_sum'
# med = 'Propofol'

# med = meds_ts_df_mrn.columns[4]
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(meds_ts_df_mrn[med])
ax[0].plot(meds_ts_df_mrn[med + '_oral'], color='green')
ax[0].plot(meds_ts_df_mrn[med + '_parenteral'], color='orange')
ax[0].plot(meds_ts_df_mrn[med + '_transdermal'], color='red')

ax[0].set_ylabel('dose (mg or mg/min)')
ax[1].scatter(meds_ts_df_mrn_hourly.index, meds_ts_df_mrn_hourly[med], color='red', s=5)
ax[1].plot(meds_ts_df_mrn_hourly.index, meds_ts_df_mrn_hourly[med], color='red', alpha = 0.4)
ax[1].set_ylabel('dose hourly sum (mg)')
ax[0].set_title(med)
# plt.savefig(str(mrn) + '_' + med + '.jpg', dpi=300)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:10: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  # Remove the CWD from sys.path while we load stuff.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0.5, 1.0, 'Propofol')

In [415]:
fig, ax = plt.subplots(3, 1, sharex=True)
ax[0].plot(meds_ts_df_mrn[med])
ax[0].set_ylabel('dose (mg or mg/min)')
ax[0].set_title(med)
ax[1].scatter(meds_ts_df_mrn_hourly.index, meds_ts_df_mrn_hourly[med], color='red', s=5)
ax[1].plot(meds_ts_df_mrn_hourly.index, meds_ts_df_mrn_hourly[med], color='red', alpha = 0.4)
ax[1].set_ylabel('dose hourly sum (mg)')
ax[2].scatter(meds_gcs_mrn_hourly.index, meds_gcs_mrn_hourly.GCS.astype(float), c='green', s=5)
ax[2].set_ylabel('GCS')

# plt.savefig(str(mrn) + '_' + med + '_gcs.jpg', dpi=300)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0, 0.5, 'GCS')

In [489]:
generic_used_benzo = [x for x in meds_ts_df_mrn_hourly.columns if ('_sum' in x) & (x.replace('_sum','') in benzos_meds)]
generic_used_opioid = [x for x in meds_ts_df_mrn_hourly.columns if ('_sum' in x) & (x.replace('_sum','') in opioids_meds)]
generic_used_antipsychotic = [x for x in meds_ts_df_mrn_hourly.columns if ('_sum' in x) & (x.replace('_sum','') in antipsychotics_meds)]

# BENZO

generic_used_benzo

def meds_plot_1(meds_ts_df_mrn_hourly, meds_list):

    fig, ax = plt.subplots(len(meds_list) + 1, 1, sharex=True, figsize=(10, max(3, len(meds_list)*1.5)))

    for i, med in enumerate(meds_list):
        
        ax[i].scatter(meds_ts_df_mrn_hourly.index, meds_ts_df_mrn_hourly[med], color='red', s=5)
        ax[i].plot(meds_ts_df_mrn_hourly.index, meds_ts_df_mrn_hourly[med], color='red', alpha = 0.4)
        ax[i].set_ylabel(med.replace('_sum', ''))
        
    ax[0].set_title(str(mrn) +' '+ title + ' hourly doses (mg)')
    ax[-1].scatter(meds_gcs_mrn_hourly.index, meds_gcs_mrn_hourly.GCS.astype(float), c='green', s=5)
    ax[-1].set_ylabel('GCS')
    
    plt.show()
    

In [490]:
generic_used_benzo

[]

In [491]:
title = 'Propopfol'
meds_plot_1(meds_ts_df_mrn_hourly, generic_used_benzo + ['Propofol'])
# plt.savefig(str(mrn) + '_' + title + '_gcs.jpg', dpi=300)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:11: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  # This is added back by InteractiveShellApp.init_path()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [56]:
meds[(meds.MRN == mrn) & (meds.MedicationDSC.apply(lambda x: 'midazolam' in x.lower()))].iloc[-5:]

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,DoseUnitCD,DoseUnitDSC,MinimumDoseAMT,MaximumDoseAMT,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,InfusionRateNBR,InfusionRateUnitCD,InfusionRateUnitDSC,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,DefinedDailyDoseNBR,MorphineEquivalentMGDoseAMT,MorphineEquivalentMGPerHourRateAMT,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,DiscreteDoseAMT,DiscreteDispenseUnitDSC,IsInfusion,Unit,Dose,MedicationDSCNew,Dose_Opioid_Equivalent,Dose_Benzo_Equivalent,Dose_Antipsychotics_Equivalent
7931,670707448.0,5236378,3.302316e+09,7005111.0,midazolam inf 5 mg/ml (100 ml) mgh,2020-04-22 12:16:00.0000000,2020-04-22 13:15:00,2020-04-24 11:27:00,43.0,mg/hr,0.0,12.0,2020-04-24 08:00:00,14.0,Rate Verify,Intravenous,11.0,0.5,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,2.5,NaN,NaN,Continuous,0-12,NaN,True,mg/hr,2.5,midazolam_inf_5_mgperml_(100_ml)_mgh,NaN,5.0,NaN
7932,670707448.0,5236378,3.302316e+09,7005111.0,midazolam inf 5 mg/ml (100 ml) mgh,2020-04-22 12:16:00.0000000,2020-04-22 13:15:00,2020-04-24 11:27:00,43.0,mg/hr,0.0,12.0,2020-04-24 09:00:00,14.0,Rate Verify,Intravenous,11.0,0.5,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,2.5,NaN,NaN,Continuous,0-12,NaN,True,mg/hr,2.5,midazolam_inf_5_mgperml_(100_ml)_mgh,NaN,5.0,NaN
7933,670707448.0,5236378,3.302316e+09,7005111.0,midazolam inf 5 mg/ml (100 ml) mgh,2020-04-22 12:16:00.0000000,2020-04-22 13:15:00,2020-04-24 11:27:00,43.0,mg/hr,0.0,12.0,2020-04-24 10:00:00,14.0,Rate Verify,Intravenous,11.0,0.5,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,2.5,NaN,NaN,Continuous,0-12,NaN,True,mg/hr,2.5,midazolam_inf_5_mgperml_(100_ml)_mgh,NaN,5.0,NaN
7934,670707448.0,5236378,3.302316e+09,7005111.0,midazolam inf 5 mg/ml (100 ml) mgh,2020-04-22 12:16:00.0000000,2020-04-22 13:15:00,2020-04-24 11:27:00,43.0,mg/hr,0.0,12.0,2020-04-24 11:00:00,14.0,Rate Verify,Intravenous,11.0,0.5,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,2.5,NaN,NaN,Continuous,0-12,NaN,True,mg/hr,2.5,midazolam_inf_5_mgperml_(100_ml)_mgh,NaN,5.0,NaN
7935,670707448.0,5236378,3.302316e+09,7005111.0,midazolam inf 5 mg/ml (100 ml) mgh,2020-04-22 12:16:00.0000000,2020-04-22 13:15:00,2020-04-24 11:27:00,43.0,mg/hr,0.0,12.0,2020-04-24 11:07:00,8.0,Stopped,Intravenous,11.0,0.0,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,0.0,NaN,NaN,Continuous,0-12,NaN,True,mg/hr,0.0,midazolam_inf_5_mgperml_(100_ml)_mgh,NaN,0.0,NaN


In [57]:
title = 'opioids'
meds_plot_1(meds_ts_df_mrn_hourly, generic_used_opioid + ['opioid_equivalent'])
plt.savefig(str(mrn) + '_' + title + '_gcs.jpg', dpi=300)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [58]:
title = 'antipsychotics'
meds_plot_1(meds_ts_df_mrn_hourly, generic_used_antipsychotic + ['antipsychotics_equivalent'])
plt.savefig(str(mrn) + '_' + title + '_gcs.jpg', dpi=300)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [59]:
generic_used_benzo

['propofol_sum', 'midazolam_sum', 'lorazepam_sum']

In [51]:
generic_used_benzo

['midazolam_sum', 'propofol_sum', 'lorazepam_sum', 'diazepam_sum']

In [46]:
benzos_ratios

{'alprazolam_oral': 0.5,
 'chlordiazepoxide_oral': 10,
 'clonazepam_oral': 0.25,
 'diazepam_oral': 5,
 'diazepam_parenteral': 5,
 'lorazepam_oral': 1,
 'lorazepam_parenteral': 1,
 'midazolam_oral': 2,
 'midazolam_parenteral': 4,
 'oxazepam_oral': 10,
 'phenobarbital_oral': 15,
 'propofol_parenteral': 160,
 'dexmedetomidine': 59.3}

In [ ]:
hdr

In [ ]:
t[0]

In [ ]:
signal_to_load_tmp

In [ ]:
meds_ts_df_mrn.head()

In [ ]:
meds_ts_df_mrn.tail()

In [ ]:
file_output_path

In [ ]:
med_name = 'hydromorphone (pf) 0.5 mg/0.5 ml injection syringe'
plt.figure()
plt.plot(meds_ts_df_mrn[med_name])

med_name = 'hydromorphone 2 mg tablet'
plt.plot(meds_ts_df_mrn[med_name])

plt.title(med_name)

In [ ]:
med_name = 'propofol 10 mg/ml intravenous emulsion'
med_name = 'opioid_equivalent'
plt.figure()
plt.plot(meds_ts_df_mrn[med_name])
plt.title(med_name)

In [ ]:
df_order

In [ ]:
meds[(meds.Unit == 'mg/hr') & (meds.IsInfusion == False)].loc[124145].OrderID

In [ ]:
meds[meds.OrderID == 462445515]

In [ ]:
antipsychotics_meds_ratios

In [ ]:
antipsychs = list(antipsychotics_meds_ratios.keys())

In [ ]:
meds[np.isin(meds.MedicationDSC, antipsychs)]['MedicationDSC'].value_counts()

In [ ]:
cohort[cohort.mrn == 5190359]

In [ ]:
meds[np.isin(meds.MedicationDSC, antipsychs)]['MRN'].value_counts();

In [ ]:
opioids_ratios, benzos_ratios, antipsychotics_ratios = general_medication_dose_equivalencies()
s_ratios

In [ ]:
opioid_meds_ratios

In [ ]:
# So, we got the doses. Let's make it a timeseries:

In [ ]:


   



#### up to here, around 95% of the entries are handled. However, some appear to have less/non-regular/missing information in them. 
# more rules:

# add unit dose to entries with missing UnitDose if the same MedicationID had a unique UnitDose in other entries.


# Weight-dependent entires, without infusion rate available. We have to multiply the OrderTXT with the weight of the patient.
# todo?


# Entries with DoseUnit (and not InfusionRate!) mL/hr: extract dose from Medicatinname, convert to mg/hr or mcg/hr here:
# ml_h_dose = df.loc[(pd.isna(df.Unit)) & \
#                               (df.DoseUnitDSC == 'mL/hr')].index
# for jloc, row in df.loc[ml_h_dose].iterrows():
#     unit_dose = re.search(r"\d+ .{1,3}/ml?", row.MedicationDSC)
#     if unit_dose is None:
#         if not row.MedicationDSC in no_dose_found:
#             no_dose_found.append(row.MedicationDSC)
#         continue
#     unit_dose = unit_dose[0]
#     assert ~pd.isna(row.DiscreteDoseAMT)
#     dose = np.float(unit_dose.split(' ')[0]) * np.float(row.DiscreteDoseAMT)
#     unit = unit_dose.split(' ')[1]
#     assert '/ml' in unit
#     unit = unit.replace('/ml', '/hr')
#     df.loc[jloc, 'Dose'] = dose
#     df.loc[jloc, 'Unit'] = unit
    

    


In [ ]:
meds.loc[(meds.DoseUnitDSC.apply(lambda x: x.lower() in weight_dependent_units)) & (pd.notna(meds.DoseUnitDSC))]

In [ ]:
row.InfusionRateUnitDSC

In [ ]:
for jloc, row in df.iterrows():
    df.DoseUnitDSC.apply(lambda x: x.lower() in weight_dependent_units)